In [0]:
select max(ingestion_date) from com_raw.kom_medical_events

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
-- Purpose:
--   Build a patient-level summary for MPS II (E761/E763) patients including:
--   - eligibility logic (Dx criteria + evidence of treatment)
--   - HCP attribution (first Dx, first Tx, latest claim, latest Tx, most-seen top 5)
--   - derived treatment timing metrics (dx->tx months, tx period months)
--   - Elaprase fill counts (windowed)
--
-- Key change implemented:
--   first_tx_after_diagnosis is ALL-TIME tx (no date filters) but must be >= incidence_date,
--   using all_tx_claims_alltime.
-- =============================================================================

CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
/* ============================================================================
   1) ELIGIBILITY COHORT BUILD
   Goal: Identify eligible MPS II patients using:
     A) "Specified" Dx (E761) with >=2 distinct Dx dates + ANY qualifying treatment evidence
     B) "Incremental Unspecified" Dx (E763) with >=2 distinct Dx dates + Elaprase-only evidence
        and NOT already included in (A)
   ========================================================================== */

-- Pull all "Specified" diagnosis events (E761) within 5-year-ish window for Dx counting.
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Specified".
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Pull all "Unspecified" diagnosis events (E763) within the same window for Dx counting.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Unspecified".
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Treatment evidence universe (broad): Elaprase NDCs OR relevant infusion/procedure codes.
-- Used to ensure "Specified" cohort has some treatment evidence in the more recent window.
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Treatment evidence (narrow): Elaprase only (NDCs + J1743).
-- Used for incremental inclusion of "Unspecified" cohort.
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Eligible "Specified" = >=2 Dx dates AND any treatment evidence.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Eligible "Incremental Unspecified" = >=2 Dx dates AND Elaprase-only evidence,
-- excluding anyone already in the specified+treatment set.
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible patient list.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

/* ============================================================================
   2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
   Goal: constrain HCPs to relevant specialties and exclude noise specialties.
   Used to filter Dx/Tx claim NPIs (but still allow NULL NPI claims through).
   ========================================================================== */
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

/* ============================================================================
   3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
   - 5Y-ish window (2020-08-01 -> end_date) used for "stats" and "latest"
   - 3Y-ish window (2022-08-01 -> end_date) used for ranking "most-seen"
   ========================================================================== */

-- All diagnosis claims (E761/E763) in the 5Y window, with NPI attribution.
-- Medical uses rendering/referring; pharmacy uses prescriber.
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Filter to "allowed" NPIs, but keep NULL NPI rows so patient-level dates won't be lost.
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- All treatment claims in the 5Y window, with a unified TX_CODE field:
--   - Elaprase NDCs from medical/pharmacy
--   - Infusion/procedure codes from medical (TX_CODE = PROCEDURE_CODE)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- NEW: ALL-TIME treatment universe (no date restriction) using same tx definition as above.
-- Used ONLY to compute "first_tx_after_diagnosis" without restricting to the 5Y window.
all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Normalize Dx + Tx into a single 5Y claim stream (TX_CODE NULL for Dx rows).
-- This enables unified "visit count" and "latest claim" logic.
all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

-- Combined Dx + Tx claims in the 3Y window for "most-seen HCP" ranking.
all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      -- Dx (medical/pharmacy)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      -- Tx (medical/pharmacy/proc)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* ============================================================================
   4) FIRST DX / FIRST TX HCP ATTRIBUTION (5Y WINDOW)
   - "first_dx_hcp": earliest Dx claim NPI per patient (ties broken by NPI)
   - "first_tx_hcp": earliest Tx claim NPI per patient (ties broken by NPI)
   - plus 5Y visit counts + last-visit dates for those attributed HCPs
   ========================================================================== */

first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),

-- Basic provider dimension for name/specialty lookup.
provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),

-- For the first Dx-attributed HCP: count all claim dates (Dx+Tx) in 5Y and get last visit.
first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),

first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),

-- For the first Tx-attributed HCP: count all claim dates (Dx+Tx) in 5Y and get last visit.
first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

-- For the first Tx-attributed HCP: count treatment claim dates only in 5Y.
first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

/* ============================================================================
   5) MOST-SEEN HCP RANKING (TOP 5) USING 3Y ACTIVITY
   - rank by #distinct visit dates in 3Y, then by recency, then by NPI
   - attach 5Y counts and 5Y last visit for those same HCPs
   ========================================================================== */

most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

/* ============================================================================
   6) HISTORICAL (ALL-TIME) FIRST DX / FIRST TX DATES (PATIENT LEVEL)
   Goal: get true first Dx date and true first Tx date without windowing.
   ========================================================================== */

historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        -- Dx specified + unspecified from both medical and pharmacy, no date filters
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),

historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        -- Tx NDCs and procedures, no date filters
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),

/* ============================================================================
   7) LATEST HCP ATTRIBUTION (5Y WINDOW)
   - latest claim HCP (across Dx+Tx): most recent claim date with an NPI
   - latest tx HCP (tx only): most recent tx claim date with an NPI
   Also compute visit counts for those attributed HCPs within the 5Y window.
   ========================================================================== */

all_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      -- Dx (medical/pharmacy)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      -- Tx (medical/pharmacy/proc)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
), 

latest_claim_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_claims_5yr
),

latest_claim_hcp as (
  select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
),

latest_claim_hcp_visit_count_5yr as (
  select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
  group by 1,2
),

latest_claim_hcp_final as (
  select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
),

-- latest_claim_hcp_ranked AS (
--     SELECT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY FILL_DATE DESC, NPI ASC
--         ) AS rn
--     FROM all_claims_5yr
--     WHERE NPI IS NOT NULL
-- ),
-- latest_claim_hcp AS (
--     SELECT
--         PATIENT_ID,
--         NPI AS latest_claim_hcp_npi,
--         FILL_DATE AS latest_claim_date
--     FROM latest_claim_hcp_ranked
--     WHERE rn = 1
-- ),
-- latest_claim_hcp_visit_count_5yr AS (
--     SELECT
--         lch.PATIENT_ID,
--         lch.latest_claim_hcp_npi,
--         COUNT(DISTINCT ac.FILL_DATE) AS latest_claim_hcp_visit_count_5yr
--     FROM latest_claim_hcp lch
--     LEFT JOIN all_claims_5yr ac
--       ON lch.PATIENT_ID = ac.PATIENT_ID
--      AND lch.latest_claim_hcp_npi = ac.NPI
--     GROUP BY lch.PATIENT_ID, lch.latest_claim_hcp_npi
-- ),

all_tx_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

most_recent_tx_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn 
  from all_tx_claims_5yr
),

most_recent_tx_hcp as (
  select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
  from most_recent_tx_hcp_ranked where rn = 1
),

latest_treatment_hcp_visit_count as (
  select a.patient_id, a.latest_treatment_hcp_npi, count(distinct fill_date) as latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a 
  left join all_tx_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
  group by 1, 2
),

most_recent_tx_hcp_final as (
  select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a
  left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
),

-- most_recent_tx_hcp_ranked AS (
--     SELECT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY FILL_DATE DESC, NPI ASC
--         ) AS rn
--     FROM all_tx_claims_5yr
--     WHERE NPI IS NOT NULL
-- ),
-- most_recent_tx_hcp AS (
--     SELECT
--         PATIENT_ID,
--         NPI AS latest_treatment_hcp_npi,
--         FILL_DATE AS latest_treatment_date
--     FROM most_recent_tx_hcp_ranked
--     WHERE rn = 1
-- ),
-- latest_treatment_hcp_visit_count AS (
--     SELECT
--         mrt.PATIENT_ID,
--         mrt.latest_treatment_hcp_npi,
--         COUNT(DISTINCT ac.FILL_DATE) AS latest_treatment_hcp_visit_count_5yr
--     FROM most_recent_tx_hcp mrt
--     LEFT JOIN all_claims_5yr ac
--       ON mrt.PATIENT_ID = ac.PATIENT_ID
--      AND mrt.latest_treatment_hcp_npi = ac.NPI
--     GROUP BY mrt.PATIENT_ID, mrt.latest_treatment_hcp_npi
-- ),

/* ============================================================================
   8) PATIENT-LEVEL "LATEST DATE" FALLBACKS (IGNORE NPI)
   Why: if claims exist but all have NULL NPI, HCP-attributed latest_* CTEs go NULL.
        These patient-level dates ensure latest_claim_date/latest_treatment_date are populated.
   ========================================================================== */

-- latest_claim_date_patient AS (
--   SELECT
--     PATIENT_ID,
--     MAX(FILL_DATE) AS latest_claim_date_any
--   FROM all_claims_5yr
--   GROUP BY PATIENT_ID
-- ),
-- latest_treatment_date_patient AS (
--   SELECT
--     PATIENT_ID,
--     MAX(FILL_DATE) AS latest_treatment_date_any
--   FROM all_tx_claims_5yr
--   GROUP BY PATIENT_ID
-- ),

/* ============================================================================
   9) LATEST TX TYPE (WINDOWED TO RECENT TREATMENT PERIOD)
   Goal: classify the latest tx within 2023-08-01..end_date as "Elaprase" vs other proc.
   ========================================================================== */

latest_mpsii_treatment_type AS (
  SELECT
    patient_id,
    CASE
      WHEN tx_code IN ('54092070001','540920700','J1743') THEN 'Elaprase'
      ELSE 'Other ERT Proc'
    END AS latest_mpsii_tx_type
  FROM (
    SELECT
      patient_id,
      fill_date,
      tx_code,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC, tx_code ASC
      ) AS rn
    FROM all_tx_claims_5yr
    WHERE tx_code IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
  )
  WHERE rn = 1
),

/* ============================================================================
   10) FIRST TX AFTER DIAGNOSIS (ALL-TIME TX, BUT MUST BE AFTER DX)
   Goal: compute earliest treatment date after incidence_date using all_tx_claims_alltime.
   ========================================================================== */

first_tx_after_diagnosis AS (
  SELECT
    tx.patient_id,
    MIN(tx.fill_date) AS first_tx_after_diagnosis
  FROM all_tx_claims_alltime tx
  INNER JOIN historical_first_dx dx
    ON tx.patient_id = dx.patient_id
  WHERE tx.fill_date >= dx.incidence_date
  GROUP BY tx.patient_id
),

/* ============================================================================
   11) ELAPRASE FILL COUNTS (WINDOWED)
   Goal: count distinct treatment dates for Elaprase-coded tx between 2023-08-01..end_date.
   ========================================================================== */

elaprase_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS elaprase_fills
  FROM all_tx_claims_5yr
  WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND tx_code IN ('54092070001','540920700','J1743')
  GROUP BY patient_id
),

/* ============================================================================
   12) PATIENT DIMENSIONS
   - demographics: pick a single record per patient
   - geography: pick "best current" state using validity logic
   ========================================================================== */

patient_demographics AS (
    SELECT *
    FROM (
        SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),
patient_geography AS (
    SELECT patient_id, patient_state
    FROM (
        SELECT
            PATIENT_ID,
            patient_state,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
                    VALID_TO_DATE DESC
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    )
    WHERE rn = 1
),

/* ============================================================================
   13) PIVOT TOP-5 MOST-SEEN HCPs INTO WIDE FORMAT
   Goal: turn rows (patient_id, rank=1..5) into columns to avoid repeated joins.
   ========================================================================== */
most_seen_pivot AS (
    SELECT
        PATIENT_ID,

        MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
        MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
        MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

        MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
        MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
        MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

        MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
        MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
        MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

        MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
        MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
        MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

        MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
        MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
        MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

    FROM most_seen_combined_stats
    GROUP BY PATIENT_ID
)

/* ============================================================================
   FINAL SELECT
   Produces one row per eligible patient with:
   - demographics + geography
   - incidence dates + latest dates (with patient-level fallbacks)
   - attributed HCPs (latest claim, latest tx, first dx, first tx, top 5 most-seen)
   - provider + HCO enrichment (reference_file)
   - derived metrics and Elaprase counts
   ========================================================================== */
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    -- Historical First Dates (ALL-TIME)
    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    -- Latest claim date: use HCP-attributed latest if available else patient-level fallback
    -- COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
    lch.latest_claim_date AS latest_claim_date,

    -- Latest claim HCP attribution (only when NPI exists on that latest claim)
    lch.latest_claim_hcp_npi,
    pdlch.provider_name     AS latest_claim_hcp_name,
    pdlch.primary_specialty AS latest_claim_hcp_specialty,
    COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

    -- Map latest-claim HCP -> HCO via reference crosswalk
    -- ref1.hco_npi  AS latest_claim_hcp_hco_npi,
    ref1.hco_name AS latest_claim_hcp_hco_name,

    -- Latest treatment date: use HCP-attributed latest if available else patient-level fallback
    mrt.latest_treatment_date AS latest_treatment_date,

    -- Latest tx type (windowed to 2023-08-01..end_date)
    lmt.latest_mpsii_tx_type,

    -- First treatment after diagnosis (ALL-TIME tx, constrained to >= incidence_date)
    fta.first_tx_after_diagnosis,

    -- Derived timing: dx -> first tx (months)
    ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
      AS time_dx_to_first_tx_in_months,

    -- Derived timing: tx period (months) = first tx after dx -> latest tx (patient-level)
    ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

    -- Elaprase fills (windowed, Elaprase-only codes)
    COALESCE(ef.elaprase_fills, 0) AS elaprase_fills,

    -- Latest treatment HCP attribution (only when NPI exists on that latest tx claim)
    mrt.latest_treatment_hcp_npi,
    pdtch.provider_name     AS latest_treatment_hcp_name,
    pdtch.primary_specialty AS latest_treatment_hcp_specialty,
    COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

    -- Map latest-tx HCP -> HCO via reference crosswalk
    -- ref2.hco_npi  AS latest_treatment_hcp_hco_npi,
    ref2.hco_name AS latest_treatment_hcp_hco_name,

    -- First Dx HCP (5Y stats)
    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

    -- First Tx HCP (5Y stats)
    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

    -- Top 5 most-seen HCPs (ranked by 3Y, with 5Y stats)
    msp.most_seen_hcp1_3yr_ranked,
    COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    msp.most_seen_hcp1_last_visit_5yr,

    msp.most_seen_hcp2_3yr_ranked,
    COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    msp.most_seen_hcp2_last_visit_5yr,

    msp.most_seen_hcp3_3yr_ranked,
    COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    msp.most_seen_hcp3_last_visit_5yr,

    msp.most_seen_hcp4_3yr_ranked,
    COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    msp.most_seen_hcp4_last_visit_5yr,

    msp.most_seen_hcp5_3yr_ranked,
    COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    msp.most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID

-- Patient-level ALL-TIME incidence dates
LEFT JOIN historical_first_dx hfdx
    ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx
    ON ep.PATIENT_ID = hftx.PATIENT_ID

-- First tx after dx (ALL-TIME)
LEFT JOIN first_tx_after_diagnosis fta
    ON ep.PATIENT_ID = fta.PATIENT_ID

-- Elaprase fills (windowed)
LEFT JOIN elaprase_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

-- Patient-level latest date fallbacks (ignore NPI)
-- LEFT JOIN latest_claim_date_patient lcd
--     ON ep.PATIENT_ID = lcd.PATIENT_ID
-- LEFT JOIN latest_treatment_date_patient ltd
--     ON ep.PATIENT_ID = ltd.PATIENT_ID

-- Latest tx type (windowed)
LEFT JOIN latest_mpsii_treatment_type lmt
    ON ep.PATIENT_ID = lmt.PATIENT_ID

-- Latest claim HCP + enrichment (provider + HCO)
LEFT JOIN latest_claim_hcp_final lch
    ON ep.PATIENT_ID = lch.PATIENT_ID
-- LEFT JOIN latest_claim_hcp_visit_count_5yr lchvc
--     ON ep.PATIENT_ID = lchvc.PATIENT_ID
--    AND lch.latest_claim_hcp_npi = lchvc.latest_claim_hcp_npi
LEFT JOIN provider_dim pdlch
    ON lch.latest_claim_hcp_npi = pdlch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
    ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- Latest tx HCP + enrichment (provider + HCO)
LEFT JOIN most_recent_tx_hcp_final mrt
    ON ep.PATIENT_ID = mrt.PATIENT_ID
-- LEFT JOIN latest_treatment_hcp_visit_count lthvc
--     ON ep.PATIENT_ID = lthvc.PATIENT_ID
--    AND mrt.latest_treatment_hcp_npi = lthvc.latest_treatment_hcp_npi
LEFT JOIN provider_dim pdtch
    ON mrt.latest_treatment_hcp_npi = pdtch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
    ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- First Dx/Tx HCP attribution + stats
LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs
    ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths
    ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
    ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- Pivoted top-5 most-seen HCPs
LEFT JOIN most_seen_pivot msp
    ON ep.PATIENT_ID = msp.PATIENT_ID

ORDER BY ep.PATIENT_ID;

-- Materialize the temp view into the persistent base table.
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;


In [0]:
-- =============================================================================
-- mpsii_tx_claims (Temp View)
--
-- What this view is:
--   A curated “treatment claims” (Tx) universe for an MPS II eligible patient cohort.
--   It outputs Tx events (medical NDC, pharmacy NDC, and procedure administrations)
--   with an attributed HCP NPI when available, within the refresh window.
--
-- Output grain:
--   One row per (patient_id, npi-attribution, fill_date) treatment event.
--   Note: NPI can be NULL (kept intentionally).
--
-- Key inputs:
--   - com_edp_prd.com_raw.kom_medical_events
--   - com_edp_prd.com_raw.kom_pharmacy_events
--   - com_raw.kom_providers  (for provider filtering)
--
-- Key logic:
--   1) Build eligible_patients using Dx evidence (E761/E763) + treatment evidence.
--   2) Define provider inclusion list (cohort_3_learnings) based on specialties.
--   3) Pull treatment events in refresh window and filter to included providers (or NULL NPI).
--
-- Parameters:
--   ${end_date} should be supplied by the runtime (Databricks widget/job parameter).
-- =============================================================================

CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
WITH
-- ============================================================================
-- 1) Dx evidence for cohort building (Specified vs Unspecified)
--    These CTEs create Dx event streams used ONLY to count distinct Dx dates.
--    Window used here: 2020-08-01 → ${end_date}
-- ============================================================================

-- Specified Dx events:
--   Pull E761 diagnosis occurrences from:
--     (a) medical_events where DIAGNOSIS_CODES contains E761, using SERVICE_DATE
--     (b) pharmacy_events where DIAGNOSIS_CODE = E761 and transaction is PAID, using FILL_DATE
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Specified Dx patients:
--   Keep patients with at least 2 distinct Dx dates (>=2 distinct fill_date values).
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Unspecified Dx events:
--   Pull E763 diagnosis occurrences from medical + paid pharmacy, same as above.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Unspecified Dx patients:
--   Keep patients with at least 2 distinct Dx dates.
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- ============================================================================
-- 2) Treatment evidence for cohort building (refresh window only)
--    These CTEs do NOT output the final Tx universe; they’re used to confirm
--    that a patient has qualifying treatment evidence in the refresh window.
--    Window used here: 2023-08-01 → ${end_date}
-- ============================================================================

-- Treatment evidence (broad):
--   Patient qualifies if they have ANY of:
--     - Elaprase NDCs in medical (NDC11) within refresh window
--     - Elaprase NDCs in pharmacy within refresh window with PAID result
--     - Any procedure/admin codes in the provided procedure list within refresh window
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Treatment evidence (Elaprase-only):
--   Narrower evidence set for incremental unspecified cohort:
--     - Elaprase NDCs (medical/pharmacy) in refresh window
--     - J1743 procedure in refresh window
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- ============================================================================
-- 3) Build eligible patient cohort
--    - Specified: >=2 E761 Dx dates AND ANY treatment evidence in refresh window
--    - Incremental unspecified: >=2 E763 Dx dates AND Elaprase-only evidence
--      AND not already in specified+treatment cohort
-- ============================================================================

-- Specified cohort:
--   Patients who meet the “>=2 specified Dx dates” requirement
--   AND have at least one qualifying treatment event in refresh window.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Incremental unspecified cohort:
--   Patients who meet the “>=2 unspecified Dx dates” requirement
--   AND have Elaprase-only evidence in refresh window
--   AND are not in the specified+treatment cohort (avoids double-counting).
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible cohort:
--   Union of specified+treatment cohort and incremental unspecified cohort.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ============================================================================
-- 4) Provider inclusion filter (cohort_3_learnings)
--    Goal: Restrict attributed NPIs to INDIVIDUAL providers that meet specialty rules.
--    This filter will be applied AFTER pulling treatment events.
--    Note: rows with NULL NPI are retained to preserve treatment dates even without attribution.
-- ============================================================================

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      -- Exclude a set of primary specialties considered out-of-scope/noise
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant',
        'Anesthesiology',
        'Dentist',
        'Dietitian, Registered',
        'Emergency Medical Technician, Basic',
        'Emergency Medicine',
        'General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered',
        'Obstetrics & Gynecology',
        'Pathology',
        'Radiology',
        'Urology'
      )
      -- OR explicitly include certain secondary specialties that are relevant
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry',
        'Psychiatry',
        'Adolescent Medicine',
        'Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine',
        'Nutrition, Pediatric',
        'Oncology, Pediatrics',
        'Pediatric Cardiology',
        'Pediatric Critical Care Medicine',
        'Pediatric Dermatology',
        'Pediatric Emergency Medicine',
        'Pediatric Endocrinology',
        'Pediatric Gastroenterology',
        'Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases',
        'Pediatric Nephrology',
        'Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery',
        'Pediatric Otolaryngology',
        'Pediatric Pulmonology',
        'Pediatric Radiology',
        'Pediatric Rehabilitation Medicine',
        'Pediatric Rheumatology',
        'Pediatric Surgery',
        'Pediatrics',
        'Clinical Biochemical Genetics',
        'Clinical Genetics (M.D.)',
        'Clinical Molecular Genetics',
        'Ph.D. Medical Genetics',
        'Neurodevelopmental Disabilities',
        'Neurology',
        'Neurology with Special Qualifications in Child Neurology',
        'Neuroradiology'
      )
    )
),

-- ============================================================================
-- 5) Treatment claims universe returned by the view (refresh/“2y” window)
--    Pull Tx events for eligible patients during 2023-08-01 → ${end_date}.
--    Sources and attribution rules:
--      A) Medical NDC events: NPI = COALESCE(rendering_npi, referring_npi), date = SERVICE_DATE
--      B) Pharmacy NDC events: NPI = prescriber_npi, date = FILL_DATE, PAID only
--      C) Procedure events:    NPI = rendering_npi, date = SERVICE_DATE
--    Then apply provider filter:
--      - keep if NPI in cohort_3_learnings OR NPI is NULL
-- ============================================================================

all_tx_claims_2yr AS (
    SELECT DISTINCT *
    FROM (
      -- Medical: Elaprase NDC events with attributed HCP (rendering/referring)
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      -- Pharmacy: Elaprase NDC fills with attributed prescriber (paid only)
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      -- Medical: procedure/admin events with attributed renderer
      SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Provider specialty filter: keep included providers OR keep NULL NPI rows
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
)

-- Final output: all treatment events in refresh window for the eligible cohort
SELECT * FROM all_tx_claims_2yr;


In [0]:
-- =============================================================================
-- most_recently_treated_hcp (Temp View)
--
-- What this view is:
--   Patient-level attribution of the “most recently treating HCP” within the
--   REFRESH treatment window (2023-08-01 → ${end_date}), plus HCP enrichment and
--   visit context metrics computed over a broader (5y-ish) claims universe.
--
-- Output grain:
--   One row per patient (only patients with an attributable treatment NPI in the
--   refresh window will appear, because latest_treating_hcp filters npi IS NOT NULL).
--
-- Key concepts:
--   - Selection window ("most recently treated"): tx_claims (refresh window)
--   - Context window ("visits / last seen"): all_claims (currently 2020-08-01 → ${end_date})
--   - Provider filter (cohort_3_learnings): keeps included INDIVIDUAL NPIs; retains NULL NPI rows
--     in claims universes, but final attribution requires non-null NPI.
--
-- Parameters:
--   ${end_date} must be provided by the runtime.
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
WITH
-- ============================================================================
-- 1) Treatment claims: refresh window source
--    This CTE simply points to the already-built tx universe from mpsii_tx_claims:
--      - eligible cohort already applied
--      - window already applied (2023-08-01 → ${end_date})
--      - provider filter already applied (allowed NPIs + NULL NPIs)
-- ============================================================================
tx_claims AS (
    SELECT DISTINCT *
    FROM mpsii_tx_claims
),

-- ============================================================================
-- 2) Re-derive eligible_patients cohort (duplicated here)
--    This block repeats the cohort logic so that the subsequent 5y Dx/Tx universes
--    (all_dx_claims_5yr / all_tx_claims_5yr) can be built inside this view.
--    NOTE: This duplication is intentional in the current script; no logic is changed.
-- ============================================================================

-- Specified Dx event stream (E761) within 2020-08-01 → ${end_date}
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Specified Dx patients: require ≥2 distinct Dx dates
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Unspecified Dx event stream (E763) within 2020-08-01 → ${end_date}
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Unspecified Dx patients: require ≥2 distinct Dx dates
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Qualifying treatment evidence (broad) in refresh window, used to ensure “active treatment”
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Elaprase-only evidence in refresh window (used only for incremental unspecified cohort)
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Specified eligible: ≥2 specified Dx dates AND any qualifying treatment in refresh window
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Incremental unspecified eligible: ≥2 unspecified Dx dates AND Elaprase-only tx in refresh window,
-- excluding already eligible specified+treatment patients
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible cohort used downstream for 5y Dx/Tx universes
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ============================================================================
-- 3) Provider filter (cohort_3_learnings)
--    Defines an allowed set of INDIVIDUAL NPIs based on specialty inclusion rules.
--    Applied to all_dx_claims_5yr / all_tx_claims_5yr, with NULL NPIs retained.
-- ============================================================================
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- ============================================================================
-- 4) Build 5y-ish claims universes for visit counts / last-visit context
--    These are bounded by 2020-08-01 → ${end_date} and filtered to eligible_patients.
--    Provider filter is applied (allowed NPIs + NULL NPIs retained).
-- ============================================================================

-- Dx claims in 5y-ish window (E761/E763)
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Tx claims in 5y-ish window (Elaprase NDCs + procedure list)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Combined claims universe used only for:
--   - counting distinct visit dates per patient↔HCP
--   - computing last observed visit date per patient↔HCP
all_claims AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),

-- ============================================================================
-- 5) Select the most recently treating HCP (refresh window)
--    Uses tx_claims (2023-08-01 → ${end_date}) to pick 1 NPI per patient:
--      - Prefer rows with non-null NPI
--      - Then pick the latest fill_date
--      - Tie-break by npi DESC (deterministic tie-breaker)
--    Final output from this CTE requires npi IS NOT NULL (attributable HCP).
-- ============================================================================
latest_treating_hcp AS (
    SELECT patient_id, npi
    FROM (
        SELECT
            patient_id,
            npi,
            fill_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY
                    CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,  -- prioritize attributed claims
                    fill_date DESC,                               -- most recent date wins
                    npi DESC                                      -- deterministic tie-break
            ) AS rn
        FROM tx_claims
    ) t
    WHERE rn = 1
      AND npi IS NOT NULL
),

-- ============================================================================
-- 6) Compute visit metrics for the selected patient↔HCP pair (5y-ish universe)
--    These metrics are NOT limited to the refresh window; they use all_claims.
-- ============================================================================

-- Count of distinct visit dates for each patient↔HCP across all_claims
visit_counts AS (
    SELECT
        patient_id,
        npi,
        COUNT(DISTINCT fill_date) AS visit_counts
    FROM all_claims
    WHERE npi IS NOT NULL
    GROUP BY patient_id, npi
),

-- Last observed visit date for each selected patient↔HCP across all_claims
last_visit_date AS (
    SELECT
        lth.patient_id,
        lth.npi,
        MAX(ac.fill_date) AS last_visit_date
    FROM latest_treating_hcp lth
    LEFT JOIN all_claims ac
      ON lth.patient_id = ac.patient_id
     AND lth.npi = ac.npi
    GROUP BY lth.patient_id, lth.npi
),

-- Attach visit metrics to the most recently treated HCP per patient
latest_treating_hcp_with_visits AS (
    SELECT
        a.patient_id,
        a.npi AS most_recently_treated_hcp,
        b.visit_counts AS no_of_visits,
        c.last_visit_date AS last_visit_date_5yr
    FROM latest_treating_hcp AS a
    LEFT JOIN visit_counts AS b
        ON a.patient_id = b.patient_id
       AND a.npi        = b.npi
    LEFT JOIN last_visit_date AS c
        ON a.patient_id = c.patient_id
       AND a.npi        = c.npi
),

-- ============================================================================
-- 7) Enrichment: provider name/specialty + HCO + territory/region mapping
--    - provider info from kom_providers (INDIVIDUAL)
--    - HCO / territory / region crosswalk from reference_file
--      with additional territory_id/region_id derived via zip_to_territory_mapping
-- ============================================================================
hcp_with_other_info AS (
    SELECT
        a.*,
        CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
        b.PRIMARY_SPECIALTY AS hcp_specialty,
        -- c.hco_npi,
        c.hco_name,
        c.territory_id,
        c.territory,
        c.region_id,
        c.region
    FROM latest_treating_hcp_with_visits AS a

    -- Provider name and specialty enrichment (limit to INDIVIDUAL provider rows)
    LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
        ON a.most_recently_treated_hcp = b.NPI
       AND b.PROVIDER_TYPE = 'INDIVIDUAL'

    -- Crosswalk HCP -> HCO and attach territory/region metadata.
    -- Inner derived tables map territory_name/region_name to numeric IDs.
    LEFT JOIN (
  SELECT
    a.hcp_npi,
    a.hco_name,
    a.territory,
    a.region,
    b.territory_id,
    c.region_id,
    a.hcp_primary_specialty AS hcp_specialty
  FROM cmpa_insights_internal_schema.reference_file a
  LEFT JOIN (
      SELECT DISTINCT territory_id, territory_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) b
    ON a.territory = b.territory_name
  LEFT JOIN (
      SELECT DISTINCT region_id, region_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) c
    ON a.region = c.region_name
) c
ON a.most_recently_treated_hcp = c.hcp_npi)
--     LEFT JOIN (
--       SELECT
--         * EXCEPT (hcp_primary_specialty),
--         hcp_primary_specialty AS hcp_specialty
--       FROM (
--         SELECT
--           a.*,
--           b.territory_id,
--           c.region_id
--         FROM cmpa_insights_internal_schema.reference_file AS a
--         LEFT JOIN (
--           SELECT DISTINCT territory_id, territory_name
--           FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--         ) AS b
--           ON a.territory = b.territory_name
--         LEFT JOIN (
--           SELECT DISTINCT region_id, region_name
--           FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--         ) AS c
--           ON a.region = c.region_name
--       )
--     ) AS c
--         ON a.most_recently_treated_hcp = c.hcp_npi
-- )

-- ============================================================================
-- FINAL SELECT
--   Renames fields to make explicit:
--     - selection window: “_2yr” (refresh window)
--     - context metrics: “_5yr” (computed from all_claims 2020-08-01 → ${end_date})
-- ============================================================================
SELECT
    patient_id,
    most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
    hcp_name AS most_recently_treated_hcp_name_2yr,

    -- Visit metrics are computed across all_claims (currently 2020-08-01 → ${end_date})
    no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
    last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

    hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
    -- hco_npi AS most_recently_treated_hcp_hco_npi,
    hco_name AS most_recently_treated_hcp_hco_name,
    territory_id AS most_recently_treated_hcp_territory_id_2yr,
    territory AS most_recently_treated_hcp_territory_2yr,
    region_id AS most_recently_treated_hcp_region_id_2yr,
    region AS most_recently_treated_hcp_region_2yr
FROM hcp_with_other_info;


In [0]:
-- =============================================================================
-- Primary HCP Assignment (Dx + Tx Claims)
--
-- Goal
--   Assign ONE “primary HCP” (NPI) per eligible MPS II patient by ranking HCPs using:
--     Tier 1) Specialty priority
--     Tier 2) Total distinct visit dates (Dx + Tx combined)
--     Tier 3) Most recent visit date
--     Tier 4) NPI tiebreaker
--
-- Date windows used in this script
--   • Dx claim extraction (“5y” universe): 2020-08-01 → ${end_date}
--   • Tx claim extraction (“5y” universe): 2020-08-01 → ${end_date}
--   • Tx eligibility window (“2y/refresh”): 2023-08-01 → ${end_date}
--
-- NPI attribution (aligned to GTM file)
--   • Medical NDC claims:      COALESCE(RENDERING_NPI, REFERRING_NPI)
--   • Medical procedure claims:RENDERING_NPI
--   • Pharmacy claims:         PRESCRIBER_NPI
--
-- Eligibility overview (eligible_patients)
--   • Specified cohort:
--       - ≥2 distinct E761 Dx dates (medical or paid pharmacy) in Dx window
--       - AND any Tx in the 2y/refresh window
--   • Incremental unspecified cohort:
--       - ≥2 distinct E763 Dx dates (medical or paid pharmacy) in Dx window
--       - AND Elaprase-coded Tx in the 2y/refresh window (NDCs or J1743)
--       - Excludes patients already in specified cohort
-- =============================================================================


-- =============================================================================
-- STEP 1: DIAGNOSIS CLAIMS (Dx universe for visit counting)
--   • Captures E761/E763 diagnosis evidence from medical + paid pharmacy events
--   • Window: 2020-08-01 → ${end_date}
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical events Dx (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Pharmacy events Dx (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (Tx universe for visit counting)
--   • Captures Elaprase-coded treatment from medical NDC, medical procedures, and paid pharmacy NDC
--   • Window: 2020-08-01 → ${end_date}
--   • Includes CODE field to retain NDC/procedure provenance
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical events Tx via NDC (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Medical events Tx via procedures (NPI = rendering_npi only)
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    PROCEDURE_CODE AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Pharmacy events Tx via NDC (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 3: Tx CLAIMS IN ELIGIBILITY WINDOW (2y/refresh)
--   • Subset of all_tx_claims restricted to 2023-08-01 → ${end_date}
--   • Used ONLY to determine cohort eligibility (not for visit counting tiers)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
--   Builds eligible_patients using Dx evidence (≥2 dates) + Tx evidence in 2y window.
-- =============================================================================

-- 4A) Specified Dx requirement: ≥2 distinct E761 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4B) Specified cohort: specified Dx + any Tx in 2y/refresh window
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental Dx requirement: ≥2 distinct E763 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4D) Elaprase-coded Tx requirement for incremental eligibility (2y/refresh window)
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('54092070001', '540920700', 'J1743');

-- 4E) Incremental cohort: incremental Dx + Elaprase-coded tx in 2y window + exclude specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) Final eligible cohort
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- Provider inclusion list (INDIVIDUAL NPIs only)
--   • Used to restrict HCPs considered for primary assignment
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
    PRIMARY_SPECIALTY NOT IN (
      'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
      'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
      'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
      'Radiology','Urology'
    )
    OR SECONDARY_SPECIALTY IN (
      'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
      'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
      'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
      'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
      'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
      'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
      'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
      'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
      'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
      'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
    )
  );


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
--   • Combines Dx + Tx events (visit dates) for eligible patients only
--   • Filters to included INDIVIDUAL NPIs (cohort_3_learnings)
--   • NOTE: As written, this excludes NULL NPI rows (because of the IN filter)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  -- Dx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  -- Tx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)
WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- =============================================================================
-- STEP 6: PRIMARY HCP ASSIGNMENT (4-tier ranking)
--   Tier 1: Specialty priority bucket (lower = better)
--   Tier 2: Total distinct visit dates (Dx + Tx)
--   Tier 3: Most recent visit date
--   Tier 4: NPI tiebreaker
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

-- Optional materialization:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
-- SELECT * FROM primary_hcp;


In [0]:
-- =============================================================================
-- STEP 7: Materialize Primary HCP table with HCP identity + HCO/territory metadata
--
-- What this step does:
--   Takes the existing primary_hcp assignment (already computed upstream)
--   and persists it as a physical table, while enriching it with:
--     - HCP full name (from kom_providers)
--     - HCO affiliation + territory/region attributes (from reference_file)
--     - territory_id / region_id (derived via zip_to_territory_mapping lookups)
--
-- What this step does NOT do:
--   - It does not re-rank or change the primary HCP selection logic.
--   - It does not filter the cohort; it simply enriches and stores primary_hcp rows.
--
-- Output:
--   com_edp_prd.cmpa_insights_internal_schema.primary_hcp
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
SELECT
    ph.*,  -- retain all columns produced by the upstream primary_hcp view/table

    -- -------------------------------------------------------------------------
    -- HCP display name enrichment
    --   Concatenate first + last name from provider dimension; COALESCE protects
    --   against nulls so string concat doesn't produce NULL.
    -- -------------------------------------------------------------------------
    COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,

    -- -------------------------------------------------------------------------
    -- HCO affiliation enrichment (via crosswalk)
    --   Map HCP NPI -> HCO NPI / HCO name using reference_file.
    -- -------------------------------------------------------------------------
    -- ref.HCO_NPI  AS primary_hcp_hco_npi_2yr,
    ref.HCO_NAME AS primary_hcp_hco_name_2yr,
    ref.HCO_CITY AS primary_hcp_hco_city_2yr,
    ref.HCO_STATE AS primary_hcp_hco_state_2yr,
    -- -------------------------------------------------------------------------
    -- Territory / region enrichment
    --   Pull both the human-readable names and numeric IDs (territory_id, region_id).
    -- -------------------------------------------------------------------------
    ref.mapped_territory_id AS primary_hcp_territory_id_2yr,
    ref.TERRITORY           AS primary_hcp_territory_2yr,
    ref.mapped_region_id    AS primary_hcp_region_id_2yr,
    ref.region              AS primary_hcp_region_2yr

FROM primary_hcp ph

-- -----------------------------------------------------------------------------
-- Join #1: Provider dimension (name enrichment)
--   Join on the attributed primary HCP NPI to retrieve FIRST_NAME / LAST_NAME.
-- -----------------------------------------------------------------------------
LEFT JOIN com_edp_prd.com_raw.kom_providers p
    ON ph.PRIMARY_HCP_NPI = p.NPI

-- -----------------------------------------------------------------------------
-- Join #2: Reference enrichment (HCO + territory + region)
--   Build a reference subquery that:
--     1) Starts from reference_file (HCP ↔ HCO + territory/region names)
--     2) Adds territory_id by mapping territory name -> territory_id via zip_to_territory_mapping
--     3) Adds region_id by mapping region name -> region_id via zip_to_territory_mapping
--     4) Renames hcp_primary_specialty to hcp_specialty (not used in final select here,
--        but retained in the ref dataset)
--   Finally join ref to primary_hcp using HCP_NPI.
-- -----------------------------------------------------------------------------
LEFT JOIN (
  SELECT
    a.hcp_npi,
    a.hco_name,
    a.hco_city,
    a.hco_state,
    a.territory,
    a.region,
    b.territory_id AS mapped_territory_id,
    c.region_id AS mapped_region_id,
    a.hcp_primary_specialty AS hcp_specialty
  FROM cmpa_insights_internal_schema.reference_file a
  LEFT JOIN (
      SELECT DISTINCT try_cast(territory_id AS BIGINT) AS territory_id, territory_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) b
    ON try_cast(a.territory_id AS BIGINT) = b.territory_id
  LEFT JOIN (
      SELECT DISTINCT try_cast(region_id AS BIGINT) AS region_id, region_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
  ) c
    ON try_cast(a.region_id AS BIGINT) = c.region_id
) ref
ON ph.PRIMARY_HCP_NPI = ref.hcp_npi;


In [0]:
-- =============================================================================
-- patient360_master (Final patient-level master table)
--
-- What this step does:
--   Materializes a single, “wide” patient-level table by combining three upstream
--   datasets into one row per patient:
--     A) patient360_base              -> core demographics + claim-derived milestones + HCP features
--     B) most_recently_treated_hcp    -> most recent treating HCP in refresh window + visit context metrics
--     C) primary_hcp                  -> primary HCP assignment + HCO/territory/region enrichment
--
-- Output:
--   com_edp_prd.cmpa_insights_internal_schema.patient360_master
--
-- Join strategy:
--   - Start from patient360_base (a) as the backbone (all patients retained)
--   - LEFT JOIN most_recently_treated_hcp (b) on patient_id to add recent-treatment attribution fields
--   - LEFT JOIN primary_hcp (c) on patient_id to add primary HCP and territory enrichment fields
--   - SELECT DISTINCT used to dedupe in case joins introduce multiplicity (e.g., enrichment tables
--     or upstream views have >1 row per patient)
--
-- Column handling:
--   - b.* EXCEPT(patient_id) and c.* EXCEPT(patient_id) avoid duplicating patient_id columns
--     from the joined datasets.
--   - Window suffixes (_2yr/_3yr/_5yr) indicate derivation windows from upstream logic.
--
-- Parameters:
--   None directly here (all parameterized logic occurs upstream), but this depends on upstream
--   tables/views that may be parameterized by ${end_date}.
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
SELECT DISTINCT
    -- -------------------------------------------------------------------------
    -- Patient identity + demographics (from patient360_base)
    -- -------------------------------------------------------------------------
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    -- -------------------------------------------------------------------------
    -- Historical milestones (patient-level; timeframe-agnostic where defined upstream)
    --   - incidence_date: earliest observed Dx date across history (as defined in base)
    --   - first_incidence_treatment_date: earliest observed treatment date across history (as defined in base)
    -- -------------------------------------------------------------------------
    a.incidence_date,
    a.first_incidence_treatment_date,

    -- -------------------------------------------------------------------------
    -- Latest claim (patient-level)
    --   - latest_claim_date may fall back to a patient-level date if NPI attribution is missing upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_date,

    -- -------------------------------------------------------------------------
    -- Latest claim attributed HCP + visit context (from patient360_base)
    --   Note: these fields are populated only if the latest claim had an attributable NPI upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,

    -- Latest claim attributed HCO (via reference enrichment in base)
    -- a.latest_claim_hcp_hco_npi,
    a.latest_claim_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- Latest treatment (patient-level) + derived treatment type
    --   - latest_treatment_date may fall back to a patient-level date if NPI attribution is missing upstream
    --   - latest_mpsii_tx_type typically derived within the refresh window upstream
    -- -------------------------------------------------------------------------
    a.latest_treatment_date,
    a.latest_mpsii_tx_type,

    -- -------------------------------------------------------------------------
    -- First Tx after Dx + timelines (from patient360_base)
    --   - first_tx_after_diagnosis is computed upstream (all-time tx constrained to >= incidence_date)
    --   - timelines are derived from incidence_date / first_tx_after_diagnosis / latest_treatment_date
    -- -------------------------------------------------------------------------
    a.first_tx_after_diagnosis,
    a.time_dx_to_first_tx_in_months,
    a.treatment_period_months,

    -- -------------------------------------------------------------------------
    -- Treatment activity proxy (from patient360_base)
    --   - elaprase_fills: distinct tx dates in refresh window as defined upstream
    -- -------------------------------------------------------------------------
    a.elaprase_fills,

    -- -------------------------------------------------------------------------
    -- Latest treatment attributed HCP + visit context (from patient360_base)
    -- -------------------------------------------------------------------------
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,

    -- Latest treatment attributed HCO (via reference enrichment in base)
    -- a.latest_treatment_hcp_hco_npi,
    a.latest_treatment_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- First Dx / Tx attributed HCP metrics (5y-ish universe; from patient360_base)
    -- -------------------------------------------------------------------------
    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Top-5 most-seen HCPs (ranked by 3y; includes 5y counts + last-visit; from patient360_base)
    -- -------------------------------------------------------------------------
    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #1: most_recently_treated_hcp
    --   Pulls in:
    --     - most_recently_treated_hcp_2yr and its attributes (name/specialty/HCO/territory/region)
    --     - visit context metrics computed over the broader claims universe in that view
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    b.* EXCEPT (patient_id),

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #2: primary_hcp
    --   Pulls in:
    --     - primary HCP attribution + name/HCO/territory/region fields
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    c.* EXCEPT (patient_id)

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base AS a

-- Left join retains all patients from patient360_base even if no match in most_recently_treated_hcp
LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id

-- Left join retains all patients from patient360_base even if no match in primary_hcp
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id;


In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
/* =====================================================================
   PURPOSE
   ---------------------------------------------------------------------
   This script derives a PATIENT-LEVEL SEVERITY classification by:
   1) Pulling diagnosis history (medical claims) for the existing patient360 cohort
   2) Flagging claim lines that contain any “severity” diagnosis codes
   3) Counting distinct service dates with severity evidence
   4) Labeling patients:
        - Severe      → 2+ distinct severity-coded diagnosis dates
        - Attenuated  → otherwise
   Notes:
   - This overwrites the existing patient360_master table with a new version
     that includes an added column: severity
   ===================================================================== */

WITH
/* ---------------------------------------------------------------------
   STEP 0: Build patient-diagnosis base table (claim-line level)
   ---------------------------------------------------------------------
   What base contains:
     - One row per patient per medical claim line (deduped)
     - Includes service_date + diagnosis_codes
   How it’s built:
     - Start from the patient universe in patient360_master (table a)
     - LEFT JOIN medical_events (table b) to bring in diagnosis history
     - Restrict medical_events to the lookback window:
         '2020-08-01' → (SELECT end_date FROM runtime_parameters)
   Why LEFT JOIN:
     - Keeps patients even if they have no qualifying medical_events rows in the window
       (they will carry NULL service_date / diagnosis_codes downstream)
   --------------------------------------------------------------------- */
base AS (
  SELECT DISTINCT
    a.PATIENT_ID,
    b.SERVICE_DATE,
    b.DIAGNOSIS_CODES
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master a
  LEFT JOIN com_edp_prd.com_raw.kom_medical_events b
    ON a.PATIENT_ID = b.PATIENT_ID
   AND b.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

/* ---------------------------------------------------------------------
   STEP 1: Flag diagnosis records with SEVERITY codes (claim-line level)
   ---------------------------------------------------------------------
   What flagged contains:
     - The same rows as base, plus a binary indicator has_severity_code
   How severity is detected:
     - DIAGNOSIS_CODES is searched using ILIKE patterns that assume pipe-delimited
       codes (e.g., “|G910|”)
     - If any of the listed codes appear in the string, has_severity_code = 1
     - Otherwise has_severity_code = 0
   Why this is done at record-level first:
     - It preserves per-date evidence so later aggregation can count distinct dates
       with severity evidence.
   --------------------------------------------------------------------- */
flagged AS (
  SELECT
    PATIENT_ID,
    SERVICE_DATE,
    CASE
      WHEN DIAGNOSIS_CODES IS NOT NULL AND (
           DIAGNOSIS_CODES ILIKE '%|G910|%' OR DIAGNOSIS_CODES ILIKE '%|G911|%' OR DIAGNOSIS_CODES ILIKE '%|G912|%'
        OR DIAGNOSIS_CODES ILIKE '%|G913|%' OR DIAGNOSIS_CODES ILIKE '%|G914|%' OR DIAGNOSIS_CODES ILIKE '%|G918|%'
        OR DIAGNOSIS_CODES ILIKE '%|G919|%' OR DIAGNOSIS_CODES ILIKE '%|Q038|%' OR DIAGNOSIS_CODES ILIKE '%|Q039|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q050|%' OR DIAGNOSIS_CODES ILIKE '%|Q051|%' OR DIAGNOSIS_CODES ILIKE '%|Q052|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q053|%' OR DIAGNOSIS_CODES ILIKE '%|Q054|%' OR DIAGNOSIS_CODES ILIKE '%|Q055|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q056|%' OR DIAGNOSIS_CODES ILIKE '%|Q057|%' OR DIAGNOSIS_CODES ILIKE '%|Q058|%'
        OR DIAGNOSIS_CODES ILIKE '%|Q0700|%' OR DIAGNOSIS_CODES ILIKE '%|Q0702|%' OR DIAGNOSIS_CODES ILIKE '%|Q0703|%'
        OR DIAGNOSIS_CODES ILIKE '%|F445|%' OR DIAGNOSIS_CODES ILIKE '%|F639|%' OR DIAGNOSIS_CODES ILIKE '%|F70|%'
        OR DIAGNOSIS_CODES ILIKE '%|F71|%' OR DIAGNOSIS_CODES ILIKE '%|F72|%' OR DIAGNOSIS_CODES ILIKE '%|F73|%'
        OR DIAGNOSIS_CODES ILIKE '%|F78|%' OR DIAGNOSIS_CODES ILIKE '%|F78A1|%' OR DIAGNOSIS_CODES ILIKE '%|F78A9|%'
        OR DIAGNOSIS_CODES ILIKE '%|F79|%' OR DIAGNOSIS_CODES ILIKE '%|F800|%' OR DIAGNOSIS_CODES ILIKE '%|F801|%'
        OR DIAGNOSIS_CODES ILIKE '%|F802|%' OR DIAGNOSIS_CODES ILIKE '%|F804|%' OR DIAGNOSIS_CODES ILIKE '%|F8081|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8082|%' OR DIAGNOSIS_CODES ILIKE '%|F8089|%' OR DIAGNOSIS_CODES ILIKE '%|F809|%'
        OR DIAGNOSIS_CODES ILIKE '%|F810|%' OR DIAGNOSIS_CODES ILIKE '%|F812|%' OR DIAGNOSIS_CODES ILIKE '%|F8181|%'
        OR DIAGNOSIS_CODES ILIKE '%|F8189|%' OR DIAGNOSIS_CODES ILIKE '%|F819|%' OR DIAGNOSIS_CODES ILIKE '%|F82|%'
        OR DIAGNOSIS_CODES ILIKE '%|F840|%' OR DIAGNOSIS_CODES ILIKE '%|F843|%' OR DIAGNOSIS_CODES ILIKE '%|F845|%'
        OR DIAGNOSIS_CODES ILIKE '%|F848|%' OR DIAGNOSIS_CODES ILIKE '%|F849|%' OR DIAGNOSIS_CODES ILIKE '%|F88|%'
        OR DIAGNOSIS_CODES ILIKE '%|F89|%' OR DIAGNOSIS_CODES ILIKE '%|R6250|%' OR DIAGNOSIS_CODES ILIKE '%|R620|%'
        OR DIAGNOSIS_CODES ILIKE '%|R6251|%' OR DIAGNOSIS_CODES ILIKE '%|R6259|%' OR DIAGNOSIS_CODES ILIKE '%|R62|%'
      )
      THEN 1 ELSE 0
    END AS has_severity_code
  FROM base
),

/* ---------------------------------------------------------------------
   STEP 2: Aggregate severity evidence to the patient level
   ---------------------------------------------------------------------
   What patient_with_severity_outcome contains:
     - One row per patient with:
         count_fill_date               = # distinct service dates observed in lookback
         severity_dx_distinct_dates    = # distinct service dates where has_severity_code=1
         severity                      = classification based on severity_dx_distinct_dates
   Classification rule:
     - Severe     if severity_dx_distinct_dates >= 2
     - Attenuated otherwise
   Notes:
     - COUNT(DISTINCT SERVICE_DATE) will ignore NULL dates.
     - The ORDER BY inside the CTE is not required for correctness, but preserves a
       deterministic output ordering if queried directly.
   --------------------------------------------------------------------- */
patient_with_severity_outcome AS (
  SELECT
    PATIENT_ID,
    COUNT(DISTINCT SERVICE_DATE) AS count_fill_date,
    COUNT(DISTINCT CASE
                     WHEN has_severity_code = 1
                     THEN SERVICE_DATE
                   END) AS severity_dx_distinct_dates,
    CASE
      WHEN COUNT(DISTINCT CASE
                            WHEN has_severity_code = 1
                            THEN SERVICE_DATE
                          END) >= 2
      THEN 'Severe'
      ELSE 'Attenuated'
    END AS severity
  FROM flagged
  GROUP BY PATIENT_ID
  ORDER BY severity_dx_distinct_dates DESC, PATIENT_ID
)

/* ---------------------------------------------------------------------
   FINAL OUTPUT
   ---------------------------------------------------------------------
   - Start from patient360_master (a) to preserve the full patient record
   - LEFT JOIN the derived severity classification (b) by PATIENT_ID
   - Result: patient360_master with an additional column: severity
   Notes:
   - Patients without any qualifying medical_events in the lookback window will
     join to NULL severity (because they won't appear in patient_with_severity_outcome
     unless base produced rows with non-null service dates).
   --------------------------------------------------------------------- */
SELECT
  a.*,
  b.severity
FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master AS a
LEFT JOIN patient_with_severity_outcome AS b
  ON a.PATIENT_ID = b.patient_id;


In [0]:
-- =============================================================================
-- patient360_master (Add comorbidity features)
--
-- What this script does:
--   Overwrites patient360_master by adding patient-level comorbidity features derived from
--   diagnosis codes (medical claims) over the lookback window 2020-08-01 → ${end_date}.
--
-- High-level flow:
--   1) Identify the patient universe (base_patients) from the current patient360_master
--   2) Pull medical events for those patients in the lookback window and explode DIAGNOSIS_CODES
--      into 1 diagnosis code per row (exploded_codes)
--   3) Map each diagnosis code into a comorbidity label via CASE logic (comorbidity_raw)
--   4) Roll comorbidities up into broader categories (comorbidity_with_category)
--   5) Aggregate to patient-level summary strings + counts (patient_with_comorbidity_outcome)
--   6) Left join those features back to patient360_master and persist the enriched table
--
-- Notes:
--   - This code assumes DIAGNOSIS_CODES uses '|' as a delimiter, and uses split/explode to parse it.
--   - Only medical events are used here (kom_medical_events), not pharmacy.
--   - CASE blocks do not have an ELSE branch; unmatched codes become NULL and are dropped downstream.
--   - Final join is LEFT JOIN so all patients remain even if they have zero mapped comorbidities.
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS
WITH
-- -----------------------------------------------------------------------------
-- Base patient universe
--   Pull distinct patients from the existing patient360_master table.
--   This defines who will be evaluated for comorbidities.
-- -----------------------------------------------------------------------------
base_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

-- -----------------------------------------------------------------------------
-- 1) Filter events + explode diagnosis codes (one code per row)
--   For each eligible patient:
--     - filter medical events to the lookback window 2020-08-01 → ${end_date}
--     - explode DIAGNOSIS_CODES into individual codes using split('|') + explode
--   Output grain:
--     - one row per (patient_id, single diagnosis code)
-- -----------------------------------------------------------------------------
exploded_codes AS (
    SELECT
        m.patient_id,
        code
    FROM com_edp_prd.com_raw.kom_medical_events m
    INNER JOIN base_patients bp
        ON m.patient_id = bp.patient_id
    LATERAL VIEW explode(split(m.DIAGNOSIS_CODES, '\\|')) s AS code
    WHERE m.SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      AND m.DIAGNOSIS_CODES IS NOT NULL
),

-- -----------------------------------------------------------------------------
-- 2) Map each individual diagnosis code → comorbidity label
--   Each WHEN clause checks whether the exploded code belongs to a predefined list
--   and assigns a comorbidity_name.
--   Output grain:
--     - one row per (patient_id, code) with a comorbidity_name when a match occurs
--   Notes:
--     - No ELSE clause: unmatched codes yield NULL comorbidity_name
--     - Empty codes are removed by the WHERE filter at the end of this CTE
-- -----------------------------------------------------------------------------
comorbidity_raw AS (
    SELECT
        patient_id,
        CASE
            --------------------------------------------------------------------
            -- Respiratory issues
            --------------------------------------------------------------------
            WHEN code IN (
                'J45909','J4520','J4530','J4540','J4541','J4531','J45901','J45998',
                'J4521','J4542','J45990','J4550','J45902','J4551','J4532','J45991'
            ) THEN 'Asthma'

            WHEN code IN ('G4733','G4730','G479','G4731','G4739')
                THEN 'Sleep Apnea'

            WHEN code IN (
                'J988','J300','J301','J302','J305','J3081','J3089','J309','J310',
                'J311','J312','J320','J321','J322','J323','J324','J328','J329',
                'J330','J331','J338','J339','J340','J341','J342','J343','J3481',
                'J348200','J348201','J348202','J348210','J348211','J348212',
                'J34829','J3489','J349','J3501','J3502','J3503','J351','J352',
                'J353','J358','J359','J36','J370','J371','J3800','J3801','J3802',
                'J381','J382','J383','J384','J385','J386','J387','J390','J391',
                'J392','J393','J398','J399'
            ) THEN 'Other diseases of upper respiratory tract'

            WHEN code IN (
                'G4700','G4734','G4761','G478','G4710','G4701','G4736','G4720',
                'G4709','G4719','G4721','G4729','G4723','G47'
            ) THEN 'Sleep related disorders'

            WHEN code IN (
                'J00','J0100','J0101','J0110','J0111','J0120','J0121','J0130',
                'J0131','J0140','J0141','J0180','J0181','J0190','J0191','J020',
                'J028','J029','J0300','J0301','J0380','J0381','J0390','J0391',
                'J040','J0410','J0411','J042','J0430','J0431','J050','J0510',
                'J0511','J060','J069'
            ) THEN 'Acute upper respiratory infections'

            --------------------------------------------------------------------
            -- Ear-related Disorders
            --------------------------------------------------------------------
            WHEN code IN (
                'H6000','H6001','H6002','H6003','H6010','H6011','H6012','H6013',
                'H6020','H6021','H6022','H60311','H60312','H60319','H60321',
                'H60322','H60329','H60391','H60392','H60399','H6040','H6041',
                'H6042','H6043','H60501','H60502','H60509','H60511','H60512',
                'H60519','H60551','H60552','H60559','H60591','H60592','H60599',
                'H6060','H6061','H6062','H608X1','H608X2','H608X9','H6090',
                'H6091','H6092','H61001','H61002','H61003','H61009','H61011',
                'H61012','H61013','H61019','H61021','H61022','H61023','H61029',
                'H61031','H61032','H61033','H61039','H61321','H61322','H61323',
                'H61329','H6240','H6241','H6242','H6500','H6501','H6502','H6504',
                'H6505','H6507','H65111','H65112','H65114','H65115','H65117',
                'H65119','H65191','H65192','H65194','H65195','H65197','H65199',
                'H6520','H6521','H6522','H6530','H6531','H6532','H65411',
                'H65412','H65419','H65491','H65492','H65499','H6590','H6591',
                'H6592','H66001','H66002','H66003','H66004','H66005','H66006',
                'H66007','H66009','H66011','H66012','H66013','H66014','H66015',
                'H66016','H66017','H66019','H6611','H6612','H6620','H6621',
                'H6622','H663X1','H663X2','H663X9','H6640','H6641','H6642',
                'H6690','H6691','H6692','H671','H672','H679','H68001','H68002',
                'H68009','H68011','H68012','H68019','H68021','H68022','H68029',
                'H70001','H70002','H70009','H70011','H70012','H70019','H70091',
                'H70092','H70099','H7010','H7011','H7012','H70201','H70202',
                'H70209','H70211','H70212','H70219','H70221','H70222','H70229',
                'H70811','H70812','H70819','H70891','H70892','H70899','H7090',
                'H7091','H7092','H7100','H7101','H7102','H7110','H7111','H7112',
                'H7120','H7121','H7122','H7130','H7131','H7132','H7190','H7191',
                'H7192','H73001','H73002','H73009','H73011','H73012','H73019',
                'H73091','H73092','H73099','H7310','H7311','H7312','H7320',
                'H7321','H7322','H7411','H7412','H7413','H7419','H7440','H7441',
                'H7442','H7443','H748X1','H748X2','H748X3','H748X9','H7490',
                'H7491','H7492','H7493','H8120','H8121','H8122','H8301','H8302',
                'H8309','H9210','H9211','H9212','H9500','H9501','H9502','H9503'
            ) THEN 'Ear Infections'

            WHEN code IN (
                'H900','H9011','H9012','H902','H903','H9041','H9042','H905',
                'H906','H9071','H9072','H908','H90A11','H90A12','H90A21',
                'H90A22','H90A31','H90A32','H9101','H9102','H9103','H9109',
                'H9120','H9121','H9122','H9123','H918X1','H918X2','H918X3',
                'H918X9','H9190','H9191','H9192','H9193','P096'
            ) THEN 'Hearing loss'

            --------------------------------------------------------------------
            -- Gastrointestinal Disorders
            --------------------------------------------------------------------
            WHEN code IN (
                'K5900','K5909','K5901','K5904','K5903','K5902','K590','K5939'
            ) THEN 'Constipation'

            WHEN code IN ('R197','K591','K580','K529')
                THEN 'Diarrhea'

            WHEN code IN (
                'K4000','K4001','K4010','K4011','K4020','K4021','K4030','K4031',
                'K4040','K4041','K4090','K4091','K450','K451','K458','K460',
                'K461','K469'
            ) THEN 'Abdominal/inguinal hernia'

            --------------------------------------------------------------------
            -- Mobility Issues
            --------------------------------------------------------------------
            WHEN code IN ('Q751','Q754','Q755')
                THEN 'Dysostosis Complex'

            WHEN code IN (
                'M2560','M25611','M25612','M25619','M25621','M25622','M25629',
                'M25631','M25632','M25639','M25641','M25642','M25649','M25651',
                'M25652','M25659','M25661','M25662','M25669','M25671','M25672',
                'M25673','M25674','M25675','M25676','M2569'
            ) THEN 'Joint Stiffness'

            WHEN code IN ('G5600','G5601','G5602','G5603')
                THEN 'Carpal tunnel syndrome'

            --------------------------------------------------------------------
            -- Neurological / Developmental
            --------------------------------------------------------------------
            WHEN code IN (
                'F05','F060','F061','F062','F0630','F0631','F0632','F0633',
                'F0634','F064','F0670','F0671','F068','F070','F0781','F0789',
                'F079','F09','F22','F23','F24','F28','F29','F3010','F3011',
                'F3012','F3013','F302','F303','F304','F308','F309','F320',
                'F321','F322','F323','F324','F325','F328','F3289','F329','F32A',
                'F330','F331','F332','F333','F3340','F3341','F3342','F338',
                'F339','F340','F348','F3481','F3489','F349','F39','F410','F411',
                'F413','F418','F419','F430','F4310','F4311','F4312','F4320',
                'F4321','F4322','F4323','F4324','F4325','F4329','F438','F4389',
                'F439','F441','F442','F450','F451','F4522','F4541','F4542',
                'F54','F59','F600','F602','F603','F604','F605','F606','F6089',
                'F609','F6381','F6389','F639','F70','F71','F72','F73','F78',
                'F78A1','F78A9','F79','F800','F801','F802','F804','F8081',
                'F8082','F8089','F809','F810','F812','F8181','F8189','F819',
                'F82','F840','F843','F845','F848','F849','F88','F89','F900',
                'F901','F902','F908','F909','F910','F911','F912','F913','F918',
                'F919','F930','F938','F939','F940','F941','F942','F948','F949',
                'F950','F951','F9821','F9829','F983','F984','F985','F988',
                'F989','F99'
            ) THEN 'Behavioral Issues'

            WHEN code IN (
                'G910','G911','G912','G913','G914','G918','G919','Q038','Q039',
                'Q050','Q051','Q052','Q053','Q054','Q055','Q056','Q057','Q058',
                'Q0700','Q0702','Q0703','F445','G40001','G40009','G40011',
                'G40019','G40101','G40109','G40111','G40119','G40201','G40209',
                'G40211','G40219','G40501','G40509','G4089','R561'
            ) THEN 'CNS Issues'

            WHEN code IN ('R6250','R620','R6252','R6251','R6259','R627','R62')
                THEN 'Lack of Physiological Development'

            --------------------------------------------------------------------
            -- Chronic conditions
            --------------------------------------------------------------------
            WHEN code IN (
                'I10','I110','I129','I130','I119','I159','I160','I158','I120',
                'I161','I1310','I150'
            ) THEN 'Hypertension'

            WHEN code IN (
                'I050','I051','I052','I058','I059','I060','I061','I062','I068',
                'I069','I070','I071','I072','I078','I079','I080','I081','I082',
                'I083','I088','I089','I340','I341','I342','I348','I3481',
                'I3489','I349','I350','I351','I352','I358','I359','I360',
                'I361','I362','I368','I369','I370','I371','I372','I378','I379'
            ) THEN 'Valvular Heart Disease'

        END AS comorbidity_name
    FROM exploded_codes
    -- Remove null/empty exploded tokens before category mapping
    WHERE code IS NOT NULL AND code <> ''
),

-- -----------------------------------------------------------------------------
-- 3) Map comorbidity label → higher-level category
--   Output grain:
--     - one row per (patient_id, comorbidity_name) with a comorbidity_category
--   Notes:
--     - Only rows with a mapped comorbidity_name are kept
--     - Category mapping is many-to-one (several comorbidities roll up to a category)
-- -----------------------------------------------------------------------------
comorbidity_with_category AS (
    SELECT
        patient_id,
        comorbidity_name,
        CASE
            WHEN comorbidity_name IN (
                'Asthma','Sleep Apnea','Other diseases of upper respiratory tract',
                'Sleep related disorders','Acute upper respiratory infections'
            ) THEN 'Respiratory issues'

            WHEN comorbidity_name IN ('Ear Infections','Hearing loss')
                THEN 'Ear-related disorders'

            WHEN comorbidity_name IN ('Constipation','Diarrhea','Abdominal/inguinal hernia')
                THEN 'Gastrointestinal disorders'

            WHEN comorbidity_name IN ('Dysostosis Complex','Joint Stiffness','Carpal tunnel syndrome')
                THEN 'Mobility issues'

            WHEN comorbidity_name IN ('Behavioral Issues','CNS Issues','Lack of Physiological Development')
                THEN 'Neurological disorders'

            WHEN comorbidity_name IN ('Hypertension','Valvular Heart Disease')
                THEN 'Other chronic conditions'
        END AS comorbidity_category
    FROM comorbidity_raw
    WHERE comorbidity_name IS NOT NULL
),

-- -----------------------------------------------------------------------------
-- 4) Patient-level aggregation
--   Produces a single row per patient with:
--     - comorbidity_categories: comma-separated unique category list
--     - count_of_comorbidity_categories: number of unique categories
--     - distinct_comorbidities: comma-separated unique comorbidity names
--     - count_of_distinct_comorbidities: number of unique comorbidity names
--   collect_set() ensures uniqueness; array_sort() makes output deterministic; array_join() flattens.
-- -----------------------------------------------------------------------------
patient_with_comorbidity_outcome as (
    SELECT
    patient_id,

    array_join(
        array_sort(collect_set(comorbidity_category)),
        ', '
    ) AS comorbidity_categories,

    size(collect_set(comorbidity_category)) AS count_of_comorbidity_categories,

    array_join(
        array_sort(collect_set(comorbidity_name)),
        ', '
    ) AS distinct_comorbidities,

    size(collect_set(comorbidity_name)) AS count_of_distinct_comorbidities

FROM comorbidity_with_category
GROUP BY patient_id
ORDER BY patient_id
)

-- -----------------------------------------------------------------------------
-- FINAL SELECT / MATERIALIZATION
--   - Start from the existing patient360_master (a) to preserve all patient features
--   - LEFT JOIN patient-level comorbidity rollup (b) to add the four new fields
--   - Persist the final enriched dataset back into patient360_master
-- -----------------------------------------------------------------------------
select
    a.*,
    b.comorbidity_categories,
    b.count_of_comorbidity_categories,
    b.distinct_comorbidities,
    b.count_of_distinct_comorbidities
from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a
left join patient_with_comorbidity_outcome as b
    on a.patient_id=b.patient_id


In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
  select distinct a.*, b.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
from com_edp_prd.cmpa_insights_internal_schema.patient360_master as a 
left join com_edp_prd.com_raw.kom_providers as b on a.PRIMARY_HCP_NPI = b.npi and b.provider_type = 'INDIVIDUAL'  

In [0]:
/* =====================================================================
   PATIENT 360 TABLE REFRESH (Enrichment Layer)
   ---------------------------------------------------------------------
   What this script does:
   - Overwrites com_edp_prd.cmpa_insights_internal_schema.patient360_master
     by enriching the existing patient360_master with:
       1) Age bucket (derived from patient_age already present in the table)
       2) Best/most current geography record (via kom_patient_geography)
       3) Zip3 (derived from patient_zip in geography)
       4) Primary + secondary payer (based on claim volume in refresh window)
       5) Active insurance group (most recent non-null insurance_group observed)
   - Treatment timelines and Elaprase metrics are assumed to already exist in
     the base patient360_master (sourced upstream).
   Parameters:
   - ${end_date} must be provided by the runtime.
   ===================================================================== */

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master AS

WITH
/* ---------------------------------------------------------------------
   Base Patient 360 table
   ---------------------------------------------------------------------
   Purpose:
   - Use the current patient360_master as the backbone of the refresh.
   - SELECT DISTINCT ensures the downstream joins start from a de-duped base.
   Output grain:
   - One row per patient (as present in patient360_master).
   --------------------------------------------------------------------- */
base_table AS (
  SELECT DISTINCT *
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

/* ---------------------------------------------------------------------
   Age Buckets
   ---------------------------------------------------------------------
   Purpose:
   - Convert patient_age into a small set of reporting-friendly buckets.
   Logic:
   - <5
   - 5–10 (inclusive)
   - 11–18 (inclusive)
   - >18
   Output grain:
   - One row per patient_id with an age_bucket label.
   --------------------------------------------------------------------- */
age_buckets AS (
  SELECT DISTINCT
    patient_id,
    CASE
      WHEN patient_age < 5 THEN '<5 years'
      WHEN patient_age BETWEEN 5 AND 10 THEN '5 - 10 years'
      WHEN patient_age BETWEEN 11 AND 16 THEN '11 - 16 years'
      ELSE '>17 years'
    END AS age_bucket
  FROM base_table
),

/* ---------------------------------------------------------------------
   Best Patient Geography Record
   ---------------------------------------------------------------------
   Purpose:
   - Choose the “best” geography row per patient from kom_patient_geography.
   Ranking logic:
   - Prefer currently active records (valid_to_date > CURRENT_DATE()) first
   - Then choose the most recent by valid_to_date DESC
   Output grain:
   - One row per patient_id (rn = 1).
   --------------------------------------------------------------------- */
patient_geography AS (
  SELECT DISTINCT *
  FROM (
    SELECT *,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY
          CASE WHEN valid_to_date > CURRENT_DATE() THEN 1 ELSE 2 END,
          valid_to_date DESC
      ) AS rn
    FROM com_edp_prd.com_raw.kom_patient_geography
  )
  WHERE rn = 1
),

/* ---------------------------------------------------------------------
   Patient Zip (Zip3)
   ---------------------------------------------------------------------
   Purpose:
   - Attach a “zip3” field to each patient from the chosen geography record.
   Notes:
   - The source column name is patient_zip; the output alias is zip3.
     (No substring is taken here; it assumes patient_zip is already zip3-formatted.)
   Output grain:
   - One row per patient_id.
   --------------------------------------------------------------------- */
patient_zip AS (
  SELECT DISTINCT
    a.patient_id,
    b.patient_zip AS zip3
  FROM base_table a
  LEFT JOIN patient_geography b
    ON a.patient_id = b.patient_id
),

/* ---------------------------------------------------------------------
   Claims-Level Payer Mapping
   ---------------------------------------------------------------------
   Purpose:
   - Create a claim-level dataset that links each claim to:
       payer_name, insurance_group (via kom_plans)
   Sources:
   - Medical events: uses service_date as fill_date and medical_event_id as claim_id
   - Pharmacy events (PAID only): uses fill_date and pharmacy_event_id as claim_id
     and selects kh_plan_id from primary/secondary plan IDs
   Output grain:
   - One row per (patient_id, claim_id) with fill_date + plan-derived payer fields.
   --------------------------------------------------------------------- */
tx_claims_payer_analysis AS (
  SELECT DISTINCT
    a.patient_id,
    a.fill_date,
    a.claim_id,
    a.kh_plan_id,
    b.payer_name,
    b.insurance_group
  FROM (
    SELECT
      patient_id,
      service_date AS fill_date,
      kh_plan_id,
      medical_event_id AS claim_id
    FROM com_edp_prd.com_raw.kom_medical_events

    UNION ALL

    SELECT
      patient_id,
      fill_date,
      COALESCE(primary_kh_plan_id, secondary_kh_plan_id) AS kh_plan_id,
      pharmacy_event_id AS claim_id
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE transaction_result = 'PAID'
  ) a
  LEFT JOIN com_edp_prd.com_raw.kom_plans b
    ON a.kh_plan_id = b.kh_plan_id
),

/* ---------------------------------------------------------------------
   Primary & Secondary Payer Assignment
   ---------------------------------------------------------------------
   Purpose:
   - Determine the top 2 payers per patient based on number of distinct claims
     in the refresh window (2023-08-01 → ${end_date}).
   Steps:
   1) Count distinct claim_id per (patient_id, payer_name) in refresh window
   2) Rank payers per patient by num_claims DESC
   3) Pivot rank 1 -> primary_payer, rank 2 -> secondary_payer
   Output grain:
   - One row per patient_id.
   --------------------------------------------------------------------- */
payer_claims_count AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY patient_id
      ORDER BY num_claims DESC
    ) AS rn
  FROM (
    SELECT
      patient_id,
      payer_name,
      COUNT(DISTINCT claim_id) AS num_claims
    FROM tx_claims_payer_analysis
    WHERE payer_name IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    GROUP BY patient_id, payer_name
  )
),

patient_primary_secondary_payer AS (
  SELECT
    patient_id,
    MAX(CASE WHEN rn = 1 THEN payer_name END) AS primary_payer,
    MAX(CASE WHEN rn = 2 THEN payer_name END) AS secondary_payer
  FROM payer_claims_count
  GROUP BY patient_id
),

/* ---------------------------------------------------------------------
   Active Insurance Group
   ---------------------------------------------------------------------
   Purpose:
   - Assign the most recent non-null insurance_group per patient.
   Logic:
   - Rank claims by fill_date DESC for each patient_id
   - Take rn = 1 to get the latest observed insurance_group
   Output grain:
   - One row per patient_id.
   --------------------------------------------------------------------- */
patient_active_insurance_group AS (
  SELECT
    patient_id,
    insurance_group AS active_insurance_group
  FROM (
    SELECT
      patient_id,
      insurance_group,
      fill_date,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC
      ) AS rn
    FROM tx_claims_payer_analysis
    WHERE insurance_group IS NOT NULL
  )
  WHERE rn = 1
)

-- =====================================================================
-- FINAL OUTPUT: patient360_master enriched with buckets + geo + payer fields
-- ---------------------------------------------------------------------
-- Join behavior:
--   - All joins are LEFT JOIN to retain every patient in base_table.
-- Added fields:
--   - age_bucket
--   - zip3
--   - primary_payer
--   - secondary_payer
--   - active_insurance_group
-- =====================================================================
SELECT
  a.*,
  b.age_bucket,
  c.zip3,
  f.primary_payer,
  f.secondary_payer,
  g.active_insurance_group

FROM base_table a
LEFT JOIN age_buckets b
  ON a.patient_id = b.patient_id
LEFT JOIN patient_zip c
  ON a.patient_id = c.patient_id
LEFT JOIN patient_primary_secondary_payer f
  ON a.patient_id = f.patient_id
LEFT JOIN patient_active_insurance_group g
  ON a.patient_id = g.patient_id;


In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
select a.*except(a.most_recently_treated_hcp_specialty_2yr, a.PRIMARY_HCP_SPECIALTY, a.primary_hcp_secondary_specialty),
b.PRIMARY_SPECIALTY as most_recently_treated_hcp_specialty_2yr, b.SECONDARY_SPECIALTY as most_recently_treated_hcp_secondary_specialty_2yr, c.PRIMARY_SPECIALTY as primary_hcp_specialty, c.SECONDARY_SPECIALTY as primary_hcp_secondary_specialty
from cmpa_insights_internal_schema.patient360_master as a
left join com_raw.kom_providers as b on a.most_recently_treated_hcp_2yr = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
left join com_raw.kom_providers as c on a.PRIMARY_HCP_NPI = c.npi and c.PROVIDER_TYPE = 'INDIVIDUAL'

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
with base_table as (
  select distinct * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
newborn_screening_flag as (
  select *,
  case when patient_state in ('IL', 'MO', 'WV', 'PA', 'KY', 'MD', 'CA', 'DE', 'FL', 'KS', 'AZ', 'MA', 'RI', 'AR', 'IA', 'NC', 'TX', 'CT') then 1 else 0 end as newborn_screening_flag
  from base_table
)
select * from newborn_screening_flag

In [0]:
-- =============================================================================
-- patient360_master (Add most recent infusion location + description)
--
-- What this script does:
--   Overwrites com_edp_prd.cmpa_insights_internal_schema.patient360_master by adding:
--     - most_recent_infusion_location     (place_of_service from the patient’s latest Tx event)
--     - most_recent_infusion_description  (human-readable description mapped from POS code)
--
-- How “most recent infusion location” is defined:
--   - Consider treatment (Tx) events in the refresh window: 2023-08-01 → ${end_date}
--   - Tx events include:
--       1) Medical events with Elaprase NDC11 (54092070001, 540920700)
--       2) Pharmacy events with Elaprase NDC11 (paid only)
--       3) Medical procedure administrations (procedure_code list)
--   - For each patient, pick the latest fill_date across these Tx events
--   - The place_of_service from that latest event is retained (blank strings are turned to NULL)
--
-- Notes / gotchas:
--   - Pharmacy events do not carry place_of_service; this script sets it to '' (blank),
--     which will become NULL after NULLIF/TRIM.
--   - If a patient’s most recent Tx event is a pharmacy claim, most_recent_infusion_location
--     will be NULL (because place_of_service is blank for pharmacy rows).
--   - The POS description join expects an integer POS code; TRY_CAST protects against
--     non-numeric values.
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.patient360_master as
WITH
-- -----------------------------------------------------------------------------
-- Base table
--   Use current patient360_master as the backbone so we only append new columns.
--   SELECT DISTINCT de-dupes the base before enrichment joins.
-- -----------------------------------------------------------------------------
base_table as (
  select distinct *
  from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),

-- -----------------------------------------------------------------------------
-- Tx patients (refresh window treatment events)
--   Build the Tx event universe for 2023-08-01 → ${end_date} capturing:
--     - patient_id
--     - attributed npi (rendering/referring/prescriber depending on source)
--     - fill_date (service_date or fill_date)
--     - place_of_service (from medical events; blank for pharmacy)
--
-- Output grain:
--   One row per distinct Tx event (deduped within each union branch).
-- -----------------------------------------------------------------------------
tx_patients as (
  -- Medical events with Elaprase NDC; capture place_of_service
  select distinct
      patient_id,
      coalesce(rendering_npi, referring_npi) as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where ndc11 in ('54092070001','540920700')
    and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

  union

  -- Pharmacy events with Elaprase NDC (PAID only); place_of_service not available → set blank
  select distinct
      patient_id,
      prescriber_npi as npi,
      fill_date,
      '' as place_of_service
  from com_edp_prd.com_raw.kom_pharmacy_events
  where ndc11 in ('54092070001','540920700')
    and transaction_result = 'PAID'
    and fill_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)

  union

  -- Medical procedure administrations; capture place_of_service
  select distinct
      patient_id,
      rendering_npi as npi,
      service_date as fill_date,
      place_of_service
  from com_edp_prd.com_raw.kom_medical_events
  where procedure_code in ('99601','99602','96365','96366','J1743','S9357','S9379',
                           '38206','38230','38232','38240','38241','38242','38243','38250')
    and service_date between '2023-08-01' and (SELECT end_date FROM runtime_parameters)
),

-- -----------------------------------------------------------------------------
-- Latest POS per patient (most recent infusion location)
--   For each patient, rank Tx events by fill_date DESC and take rn = 1.
--   Clean-up:
--     - TRIM whitespace
--     - NULLIF converts blank strings to NULL
-- Output grain:
--   One row per patient_id with the most recent POS code (as a string).
-- -----------------------------------------------------------------------------
latest_pos_patients as (
  select
      patient_id,
      nullif(trim(place_of_service), '') as most_recent_infusion_location
  from (
    select
        patient_id,
        place_of_service,
        fill_date,
        row_number() over (
          partition by patient_id
          order by fill_date desc
        ) as rn
    from tx_patients
  ) t
  where rn = 1
)

-- -----------------------------------------------------------------------------
-- FINAL SELECT / MATERIALIZATION
--   - Start from base_table to retain all existing patient360 fields
--   - LEFT JOIN latest POS enrichment (may be NULL if no Tx events or pharmacy-most-recent)
--   - LEFT JOIN POS description mapping:
--       * Convert POS code string to INT using TRY_CAST
--       * Join to pos_description.code to retrieve human-readable description
-- -----------------------------------------------------------------------------
select
    a.*,
    b.most_recent_infusion_location,
    c.description as most_recent_infusion_description
from base_table a
left join latest_pos_patients b
  on a.patient_id = b.patient_id
left join com_edp_prd.cmpa_insights_internal_schema.pos_description c
  on try_cast(nullif(trim(b.most_recent_infusion_location), '') as int) = c.code


# Tivi Colums

In [0]:
%sql
select max(ingestion_date) from com_raw.kom_medical_events;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
%sql 
-- =============================================================================
-- Patient HCP Visit Summary View - Top 5 based on 3-Year activity
-- Purpose:
--   Build a patient-level summary for MPS II (E761/E763) patients including:
--   - eligibility logic (Dx criteria + evidence of treatment)
--   - HCP attribution (first Dx, first Tx, latest claim, latest Tx, most-seen top 5)
--   - derived treatment timing metrics (dx->tx months, tx period months)
--   - Tivi fill counts (windowed)
--
-- Key change implemented:
--   first_tx_after_diagnosis is ALL-TIME tx (no date filters) but must be >= incidence_date,
--   using all_tx_claims_alltime.
-- =============================================================================

CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary AS
WITH
/* ============================================================================
   1) ELIGIBILITY COHORT BUILD
   Goal: Identify eligible MPS II patients using:
     A) "Specified" Dx (E761) with >=2 distinct Dx dates + ANY qualifying treatment evidence
     B) "Incremental Unspecified" Dx (E763) with >=2 distinct Dx dates + Tivi-only evidence
        and NOT already included in (A)
   ========================================================================== */

-- Pull all "Specified" diagnosis events (E761) within 5-year-ish window for Dx counting.
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Specified".
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Pull all "Unspecified" diagnosis events (E763) within the same window for Dx counting.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Unspecified".
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Treatment evidence universe (broad): Tivi NDCs OR relevant infusion/procedure codes.
-- Used to ensure "Specified" cohort has some treatment evidence in the more recent window.
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Treatment evidence (narrow): Tivi only (NDCs + J1743).
-- Used for incremental inclusion of "Unspecified" cohort.
MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE = 'J1743'
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Eligible "Specified" = >=2 Dx dates AND any treatment evidence.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Eligible "Incremental Unspecified" = >=2 Dx dates AND Tivi-only evidence,
-- excluding anyone already in the specified+treatment set.
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible patient list.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

/* ============================================================================
   2) PROVIDER FILTER ("COHORT 3 LEARNINGS")
   Goal: constrain HCPs to relevant specialties and exclude noise specialties.
   Used to filter Dx/Tx claim NPIs (but still allow NULL NPI claims through).
   ========================================================================== */
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

/* ============================================================================
   3) CLAIMS UNIVERSES (DX + TX) WITH NPI ATTRIBUTION
   - 5Y-ish window (2020-08-01 -> end_date) used for "stats" and "latest"
   - 3Y-ish window (2022-08-01 -> end_date) used for ranking "most-seen"
   ========================================================================== */

-- All diagnosis claims (E761/E763) in the 5Y window, with NPI attribution.
-- Medical uses rendering/referring; pharmacy uses prescriber.
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Filter to "allowed" NPIs, but keep NULL NPI rows so patient-level dates won't be lost.
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- All treatment claims in the 5Y window, with a unified TX_CODE field:
--   - Tivi NDCs from medical/pharmacy
--   - Infusion/procedure codes from medical (TX_CODE = PROCEDURE_CODE)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- SELECT DISTINCT
      --     PATIENT_ID,
      --     RENDERING_NPI AS NPI,
      --     SERVICE_DATE AS FILL_DATE,
      --     PROCEDURE_CODE AS TX_CODE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- NEW: ALL-TIME treatment universe (no date restriction) using same tx definition as above.
-- Used ONLY to compute "first_tx_after_diagnosis" without restricting to the 5Y window.
all_tx_claims_alltime AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- SELECT DISTINCT
      --     PATIENT_ID,
      --     RENDERING_NPI AS NPI,
      --     SERVICE_DATE AS FILL_DATE,
      --     PROCEDURE_CODE AS TX_CODE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Normalize Dx + Tx into a single 5Y claim stream (TX_CODE NULL for Dx rows).
-- This enables unified "visit count" and "latest claim" logic.
all_claims_5yr AS (
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    UNION
    SELECT DISTINCT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        TX_CODE
    FROM all_tx_claims_5yr
),

-- Combined Dx + Tx claims in the 3Y window for "most-seen HCP" ranking.
all_claims_3yr AS (
    SELECT DISTINCT *
    FROM (
      -- Dx (medical/pharmacy)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      -- Tx (medical/pharmacy/proc)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      -- UNION
      -- SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2022-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

/* ============================================================================
   4) FIRST DX / FIRST TX HCP ATTRIBUTION (5Y WINDOW)
   - "first_dx_hcp": earliest Dx claim NPI per patient (ties broken by NPI)
   - "first_tx_hcp": earliest Tx claim NPI per patient (ties broken by NPI)
   - plus 5Y visit counts + last-visit dates for those attributed HCPs
   ========================================================================== */

first_dx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_dx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_dx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_dx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_dx_hcp, first_dx_date
    FROM first_dx_hcp_ranked
    WHERE rn = 1
),

-- Basic provider dimension for name/specialty lookup.
provider_dim AS (
    SELECT
        npi,
        CONCAT(FIRST_NAME, ' ', LAST_NAME) AS provider_name,
        primary_specialty
    FROM com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
),

-- For the first Dx-attributed HCP: count all claim dates (Dx+Tx) in 5Y and get last visit.
first_dx_hcp_5yr_stats AS (
    SELECT fdh.PATIENT_ID, fdh.first_dx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_dx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_dx_last_visit_5yr
    FROM first_dx_hcp fdh
    LEFT JOIN all_claims_5yr ac
      ON fdh.PATIENT_ID = ac.PATIENT_ID AND fdh.first_dx_hcp = ac.NPI
    GROUP BY fdh.PATIENT_ID, fdh.first_dx_hcp
),

first_tx_hcp_ranked AS (
    SELECT PATIENT_ID, NPI, MIN(FILL_DATE) AS first_tx_date,
           ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY MIN(FILL_DATE) ASC, NPI) AS rn
    FROM all_tx_claims_5yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),
first_tx_hcp AS (
    SELECT PATIENT_ID, NPI AS first_tx_hcp, first_tx_date
    FROM first_tx_hcp_ranked
    WHERE rn = 1
),

-- For the first Tx-attributed HCP: count all claim dates (Dx+Tx) in 5Y and get last visit.
first_tx_hcp_5yr_stats AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT ac.FILL_DATE) AS first_tx_all_visit_count_5yr,
           MAX(ac.FILL_DATE) AS first_tx_last_visit_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_claims_5yr ac
      ON fth.PATIENT_ID = ac.PATIENT_ID AND fth.first_tx_hcp = ac.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

-- For the first Tx-attributed HCP: count treatment claim dates only in 5Y.
first_tx_hcp_5yr_tx_only AS (
    SELECT fth.PATIENT_ID, fth.first_tx_hcp,
           COUNT(DISTINCT tx.FILL_DATE) AS first_tx_treatment_visit_count_5yr
    FROM first_tx_hcp fth
    LEFT JOIN all_tx_claims_5yr tx
      ON fth.PATIENT_ID = tx.PATIENT_ID AND fth.first_tx_hcp = tx.NPI
    GROUP BY fth.PATIENT_ID, fth.first_tx_hcp
),

/* ============================================================================
   5) MOST-SEEN HCP RANKING (TOP 5) USING 3Y ACTIVITY
   - rank by #distinct visit dates in 3Y, then by recency, then by NPI
   - attach 5Y counts and 5Y last visit for those same HCPs
   ========================================================================== */

most_seen_3yr_ranking AS (
    SELECT PATIENT_ID,
           NPI,
           COUNT(DISTINCT FILL_DATE) AS visit_count_3yr,
           MAX(FILL_DATE) AS last_visit_3yr,
           ROW_NUMBER() OVER (
             PARTITION BY PATIENT_ID
             ORDER BY COUNT(DISTINCT FILL_DATE) DESC,
                      MAX(FILL_DATE) DESC,
                      NPI ASC
           ) AS rank
    FROM all_claims_3yr
    WHERE NPI IS NOT NULL
    GROUP BY PATIENT_ID, NPI
),

most_seen_combined_stats AS (
    SELECT
        ms3.PATIENT_ID,
        ms3.NPI,
        ms3.rank,
        ms3.visit_count_3yr,
        ms3.last_visit_3yr,
        COUNT(DISTINCT ac5.FILL_DATE) AS visit_count_5yr,
        MAX(ac5.FILL_DATE)          AS last_visit_5yr
    FROM most_seen_3yr_ranking ms3
    LEFT JOIN all_claims_5yr ac5
      ON ms3.PATIENT_ID = ac5.PATIENT_ID AND ms3.NPI = ac5.NPI
    WHERE ms3.rank <= 5
    GROUP BY ms3.PATIENT_ID, ms3.NPI, ms3.rank, ms3.visit_count_3yr, ms3.last_visit_3yr
),

/* ============================================================================
   6) HISTORICAL (ALL-TIME) FIRST DX / FIRST TX DATES (PATIENT LEVEL)
   Goal: get true first Dx date and true first Tx date without windowing.
   ========================================================================== */

historical_first_dx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS incidence_date
    FROM (
        -- Dx specified + unspecified from both medical and pharmacy, no date filters
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E761%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E761'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE DIAGNOSIS_CODES LIKE '%E763%'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE DIAGNOSIS_CODE = 'E763'
          AND TRANSACTION_STATUS = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_dx
    GROUP BY PATIENT_ID
),

historical_first_tx AS (
    SELECT PATIENT_ID, MIN(FILL_DATE) AS first_incidence_treatment_date
    FROM (
        -- Tx NDCs and procedures, no date filters
        SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        UNION ALL
        SELECT DISTINCT PATIENT_ID, FILL_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    ) all_tx
    GROUP BY PATIENT_ID
),

/* ============================================================================
   7) LATEST HCP ATTRIBUTION (5Y WINDOW)
   - latest claim HCP (across Dx+Tx): most recent claim date with an NPI
   - latest tx HCP (tx only): most recent tx claim date with an NPI
   Also compute visit counts for those attributed HCPs within the 5Y window.
   ========================================================================== */

all_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      -- Dx (medical/pharmacy)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION
      -- Tx (medical/pharmacy/proc)
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      -- UNION
      -- SELECT DISTINCT PATIENT_ID, RENDERING_NPI AS NPI, SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
), 

latest_claim_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn
  from all_claims_5yr
),

latest_claim_hcp as (
  select patient_id, npi as latest_claim_hcp_npi, fill_date as latest_claim_date from latest_claim_hcp_ranked where rn = 1
),

latest_claim_hcp_visit_count_5yr as (
  select a.patient_id, a.latest_claim_hcp_npi, count(distinct fill_date) as latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join all_claims_5yr as b on a.patient_id = b.patient_id and a.latest_claim_hcp_npi = b.npi
  group by 1,2
),

latest_claim_hcp_final as (
  select a.patient_id, a.latest_claim_hcp_npi, a.latest_claim_date, b.latest_claim_hcp_visit_count_5yr
  from latest_claim_hcp as a
  left join latest_claim_hcp_visit_count_5yr as b on a.patient_id = b.patient_id
),

-- latest_claim_hcp_ranked AS (
--     SELECT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY FILL_DATE DESC, NPI ASC
--         ) AS rn
--     FROM all_claims_5yr
--     WHERE NPI IS NOT NULL
-- ),
-- latest_claim_hcp AS (
--     SELECT
--         PATIENT_ID,
--         NPI AS latest_claim_hcp_npi,
--         FILL_DATE AS latest_claim_date
--     FROM latest_claim_hcp_ranked
--     WHERE rn = 1
-- ),
-- latest_claim_hcp_visit_count_5yr AS (
--     SELECT
--         lch.PATIENT_ID,
--         lch.latest_claim_hcp_npi,
--         COUNT(DISTINCT ac.FILL_DATE) AS latest_claim_hcp_visit_count_5yr
--     FROM latest_claim_hcp lch
--     LEFT JOIN all_claims_5yr ac
--       ON lch.PATIENT_ID = ac.PATIENT_ID
--      AND lch.latest_claim_hcp_npi = ac.NPI
--     GROUP BY lch.PATIENT_ID, lch.latest_claim_hcp_npi
-- ),

all_tx_claims_5yr_specialty_removed AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- SELECT DISTINCT
      --     PATIENT_ID,
      --     RENDERING_NPI AS NPI,
      --     SERVICE_DATE AS FILL_DATE,
      --     PROCEDURE_CODE AS TX_CODE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

most_recent_tx_hcp_ranked as (
  select patient_id, npi, fill_date, row_number() over(partition by patient_id order by fill_date desc, npi asc) as rn 
  from all_tx_claims_5yr
),

most_recent_tx_hcp as (
  select patient_id, npi as latest_treatment_hcp_npi, fill_date as latest_treatment_date
  from most_recent_tx_hcp_ranked where rn = 1
),

latest_treatment_hcp_visit_count as (
  select a.patient_id, a.latest_treatment_hcp_npi, count(distinct fill_date) as latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a 
  left join all_tx_claims_5yr as b on a.patient_id = b.patient_id and a.latest_treatment_hcp_npi = b.npi
  group by 1, 2
),

most_recent_tx_hcp_final as (
  select a.patient_id, a.latest_treatment_hcp_npi, a.latest_treatment_date, b.latest_treatment_hcp_visit_count_5yr
  from most_recent_tx_hcp as a
  left join latest_treatment_hcp_visit_count as b on a.patient_id = b.patient_id
),

-- most_recent_tx_hcp_ranked AS (
--     SELECT
--         PATIENT_ID,
--         NPI,
--         FILL_DATE,
--         ROW_NUMBER() OVER (
--             PARTITION BY PATIENT_ID
--             ORDER BY FILL_DATE DESC, NPI ASC
--         ) AS rn
--     FROM all_tx_claims_5yr
--     WHERE NPI IS NOT NULL
-- ),
-- most_recent_tx_hcp AS (
--     SELECT
--         PATIENT_ID,
--         NPI AS latest_treatment_hcp_npi,
--         FILL_DATE AS latest_treatment_date
--     FROM most_recent_tx_hcp_ranked
--     WHERE rn = 1
-- ),
-- latest_treatment_hcp_visit_count AS (
--     SELECT
--         mrt.PATIENT_ID,
--         mrt.latest_treatment_hcp_npi,
--         COUNT(DISTINCT ac.FILL_DATE) AS latest_treatment_hcp_visit_count_5yr
--     FROM most_recent_tx_hcp mrt
--     LEFT JOIN all_claims_5yr ac
--       ON mrt.PATIENT_ID = ac.PATIENT_ID
--      AND mrt.latest_treatment_hcp_npi = ac.NPI
--     GROUP BY mrt.PATIENT_ID, mrt.latest_treatment_hcp_npi
-- ),

/* ============================================================================
   8) PATIENT-LEVEL "LATEST DATE" FALLBACKS (IGNORE NPI)
   Why: if claims exist but all have NULL NPI, HCP-attributed latest_* CTEs go NULL.
        These patient-level dates ensure latest_claim_date/latest_treatment_date are populated.
   ========================================================================== */

-- latest_claim_date_patient AS (
--   SELECT
--     PATIENT_ID,
--     MAX(FILL_DATE) AS latest_claim_date_any
--   FROM all_claims_5yr
--   GROUP BY PATIENT_ID
-- ),
-- latest_treatment_date_patient AS (
--   SELECT
--     PATIENT_ID,
--     MAX(FILL_DATE) AS latest_treatment_date_any
--   FROM all_tx_claims_5yr
--   GROUP BY PATIENT_ID
-- ),

/* ============================================================================
   9) LATEST TX TYPE (WINDOWED TO RECENT TREATMENT PERIOD)
   Goal: classify the latest tx within 2023-08-01..end_date as "Tivi" vs other proc.
   ========================================================================== */

latest_mpsii_treatment_type AS (
  SELECT
    patient_id,
    CASE
      WHEN tx_code IN ('8497600101') THEN 'Tivi - Avalayah'
    END AS latest_mpsii_tx_type
  FROM (
    SELECT
      patient_id,
      fill_date,
      tx_code,
      ROW_NUMBER() OVER (
        PARTITION BY patient_id
        ORDER BY fill_date DESC, tx_code ASC
      ) AS rn
    FROM all_tx_claims_5yr
    WHERE tx_code IS NOT NULL
      AND fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
  )
  WHERE rn = 1
),

/* ============================================================================
   10) FIRST TX AFTER DIAGNOSIS (ALL-TIME TX, BUT MUST BE AFTER DX)
   Goal: compute earliest treatment date after incidence_date using all_tx_claims_alltime.
   ========================================================================== */

first_tx_after_diagnosis AS (
  SELECT
    tx.patient_id,
    MIN(tx.fill_date) AS first_tx_after_diagnosis
  FROM all_tx_claims_alltime tx
  INNER JOIN historical_first_dx dx
    ON tx.patient_id = dx.patient_id
  WHERE tx.fill_date >= dx.incidence_date
  GROUP BY tx.patient_id
),

/* ============================================================================
   11) Tivi FILL COUNTS (WINDOWED)
   Goal: count distinct treatment dates for Tivi-coded tx between 2023-08-01..end_date.
   ========================================================================== */

Tivi_fills AS (
  SELECT
    patient_id,
    COUNT(DISTINCT fill_date) AS Tivi_fills
  FROM all_tx_claims_5yr
  WHERE fill_date BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    AND tx_code IN ('8497600101')
  GROUP BY patient_id
),

/* ============================================================================
   12) PATIENT DIMENSIONS
   - demographics: pick a single record per patient
   - geography: pick "best current" state using validity logic
   ========================================================================== */

patient_demographics AS (
    SELECT *
    FROM (
        SELECT DISTINCT PATIENT_ID, PATIENT_YOB, PATIENT_GENDER,
               ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY PATIENT_YOB ASC) AS rn
        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),
patient_geography AS (
    SELECT patient_id, patient_state
    FROM (
        SELECT
            PATIENT_ID,
            patient_state,
            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY
                    CASE WHEN VALID_TO_DATE > CURRENT_DATE() THEN 1 ELSE 2 END,
                    VALID_TO_DATE DESC
            ) AS rn
        FROM COM_EDP_PRD.COM_RAW.KOM_PATIENT_GEOGRAPHY
    )
    WHERE rn = 1
),

/* ============================================================================
   13) PIVOT TOP-5 MOST-SEEN HCPs INTO WIDE FORMAT
   Goal: turn rows (patient_id, rank=1..5) into columns to avoid repeated joins.
   ========================================================================== */
most_seen_pivot AS (
    SELECT
        PATIENT_ID,

        MAX(CASE WHEN rank = 1 THEN NPI END)              AS most_seen_hcp1_3yr_ranked,
        MAX(CASE WHEN rank = 1 THEN visit_count_5yr END)  AS most_seen_hcp1_visit_count_5yr,
        MAX(CASE WHEN rank = 1 THEN last_visit_5yr END)   AS most_seen_hcp1_last_visit_5yr,

        MAX(CASE WHEN rank = 2 THEN NPI END)              AS most_seen_hcp2_3yr_ranked,
        MAX(CASE WHEN rank = 2 THEN visit_count_5yr END)  AS most_seen_hcp2_visit_count_5yr,
        MAX(CASE WHEN rank = 2 THEN last_visit_5yr END)   AS most_seen_hcp2_last_visit_5yr,

        MAX(CASE WHEN rank = 3 THEN NPI END)              AS most_seen_hcp3_3yr_ranked,
        MAX(CASE WHEN rank = 3 THEN visit_count_5yr END)  AS most_seen_hcp3_visit_count_5yr,
        MAX(CASE WHEN rank = 3 THEN last_visit_5yr END)   AS most_seen_hcp3_last_visit_5yr,

        MAX(CASE WHEN rank = 4 THEN NPI END)              AS most_seen_hcp4_3yr_ranked,
        MAX(CASE WHEN rank = 4 THEN visit_count_5yr END)  AS most_seen_hcp4_visit_count_5yr,
        MAX(CASE WHEN rank = 4 THEN last_visit_5yr END)   AS most_seen_hcp4_last_visit_5yr,

        MAX(CASE WHEN rank = 5 THEN NPI END)              AS most_seen_hcp5_3yr_ranked,
        MAX(CASE WHEN rank = 5 THEN visit_count_5yr END)  AS most_seen_hcp5_visit_count_5yr,
        MAX(CASE WHEN rank = 5 THEN last_visit_5yr END)   AS most_seen_hcp5_last_visit_5yr

    FROM most_seen_combined_stats
    GROUP BY PATIENT_ID
)

/* ============================================================================
   FINAL SELECT
   Produces one row per eligible patient with:
   - demographics + geography
   - incidence dates + latest dates (with patient-level fallbacks)
   - attributed HCPs (latest claim, latest tx, first dx, first tx, top 5 most-seen)
   - provider + HCO enrichment (reference_file)
   - derived metrics and Tivi counts
   ========================================================================== */
SELECT
    ep.PATIENT_ID,
    pd.PATIENT_YOB,
    YEAR(CURRENT_DATE) - YEAR(pd.PATIENT_YOB) AS PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.patient_state,

    -- Historical First Dates (ALL-TIME)
    hfdx.incidence_date,
    hftx.first_incidence_treatment_date,

    -- Latest claim date: use HCP-attributed latest if available else patient-level fallback
    -- COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
    lch.latest_claim_date AS latest_claim_date,

    -- Latest claim HCP attribution (only when NPI exists on that latest claim)
    lch.latest_claim_hcp_npi,
    pdlch.provider_name     AS latest_claim_hcp_name,
    pdlch.primary_specialty AS latest_claim_hcp_specialty,
    COALESCE(lch.latest_claim_hcp_visit_count_5yr, 0) AS latest_claim_hcp_visit_count,

    -- Map latest-claim HCP -> HCO via reference crosswalk
    -- ref1.hco_npi  AS latest_claim_hcp_hco_npi,
    ref1.hco_name AS latest_claim_hcp_hco_name,

    -- Latest treatment date: use HCP-attributed latest if available else patient-level fallback
    mrt.latest_treatment_date AS latest_treatment_date,

    -- Latest tx type (windowed to 2023-08-01..end_date)
    lmt.latest_mpsii_tx_type,

    -- First treatment after diagnosis (ALL-TIME tx, constrained to >= incidence_date)
    fta.first_tx_after_diagnosis,

    -- Derived timing: dx -> first tx (months)
    ROUND(MONTHS_BETWEEN(fta.first_tx_after_diagnosis, hfdx.incidence_date), 0)
      AS time_dx_to_first_tx_in_months,

    -- Derived timing: tx period (months) = first tx after dx -> latest tx (patient-level)
    ROUND(MONTHS_BETWEEN(mrt.latest_treatment_date, fta.first_tx_after_diagnosis), 0) AS treatment_period_months,

    -- Tivi fills (windowed, Tivi-only codes)
    COALESCE(ef.Tivi_fills, 0) AS Tivi_fills,

    -- Latest treatment HCP attribution (only when NPI exists on that latest tx claim)
    mrt.latest_treatment_hcp_npi,
    pdtch.provider_name     AS latest_treatment_hcp_name,
    pdtch.primary_specialty AS latest_treatment_hcp_specialty,
    COALESCE(mrt.latest_treatment_hcp_visit_count_5yr, 0) AS latest_treatment_hcp_visit_count,

    -- Map latest-tx HCP -> HCO via reference crosswalk
    -- ref2.hco_npi  AS latest_treatment_hcp_hco_npi,
    ref2.hco_name AS latest_treatment_hcp_hco_name,

    -- First Dx HCP (5Y stats)
    fdh.first_dx_hcp AS first_dx_hcp_5yr,
    COALESCE(fdhs.first_dx_all_visit_count_5yr, 0) AS first_dx_all_visit_count_5yr,
    fdhs.first_dx_last_visit_5yr AS first_dx_last_visit_5yr,

    -- First Tx HCP (5Y stats)
    fth.first_tx_hcp AS first_tx_hcp_5yr,
    COALESCE(fths.first_tx_all_visit_count_5yr, 0) AS first_tx_all_visit_count_5yr,
    COALESCE(fthtx.first_tx_treatment_visit_count_5yr, 0) AS first_tx_treatment_visit_count_5yr,
    fths.first_tx_last_visit_5yr AS first_tx_last_visit_5yr,

    -- Top 5 most-seen HCPs (ranked by 3Y, with 5Y stats)
    msp.most_seen_hcp1_3yr_ranked,
    COALESCE(msp.most_seen_hcp1_visit_count_5yr, 0) AS most_seen_hcp1_visit_count_5yr,
    msp.most_seen_hcp1_last_visit_5yr,

    msp.most_seen_hcp2_3yr_ranked,
    COALESCE(msp.most_seen_hcp2_visit_count_5yr, 0) AS most_seen_hcp2_visit_count_5yr,
    msp.most_seen_hcp2_last_visit_5yr,

    msp.most_seen_hcp3_3yr_ranked,
    COALESCE(msp.most_seen_hcp3_visit_count_5yr, 0) AS most_seen_hcp3_visit_count_5yr,
    msp.most_seen_hcp3_last_visit_5yr,

    msp.most_seen_hcp4_3yr_ranked,
    COALESCE(msp.most_seen_hcp4_visit_count_5yr, 0) AS most_seen_hcp4_visit_count_5yr,
    msp.most_seen_hcp4_last_visit_5yr,

    msp.most_seen_hcp5_3yr_ranked,
    COALESCE(msp.most_seen_hcp5_visit_count_5yr, 0) AS most_seen_hcp5_visit_count_5yr,
    msp.most_seen_hcp5_last_visit_5yr

FROM eligible_patients ep
LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID
LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID

-- Patient-level ALL-TIME incidence dates
LEFT JOIN historical_first_dx hfdx
    ON ep.PATIENT_ID = hfdx.PATIENT_ID
LEFT JOIN historical_first_tx hftx
    ON ep.PATIENT_ID = hftx.PATIENT_ID

-- First tx after dx (ALL-TIME)
LEFT JOIN first_tx_after_diagnosis fta
    ON ep.PATIENT_ID = fta.PATIENT_ID

-- Tivi fills (windowed)
LEFT JOIN Tivi_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

-- Patient-level latest date fallbacks (ignore NPI)
-- LEFT JOIN latest_claim_date_patient lcd
--     ON ep.PATIENT_ID = lcd.PATIENT_ID
-- LEFT JOIN latest_treatment_date_patient ltd
--     ON ep.PATIENT_ID = ltd.PATIENT_ID

-- Latest tx type (windowed)
LEFT JOIN latest_mpsii_treatment_type lmt
    ON ep.PATIENT_ID = lmt.PATIENT_ID

-- Latest claim HCP + enrichment (provider + HCO)
LEFT JOIN latest_claim_hcp_final lch
    ON ep.PATIENT_ID = lch.PATIENT_ID
-- LEFT JOIN latest_claim_hcp_visit_count_5yr lchvc
--     ON ep.PATIENT_ID = lchvc.PATIENT_ID
--    AND lch.latest_claim_hcp_npi = lchvc.latest_claim_hcp_npi
LEFT JOIN provider_dim pdlch
    ON lch.latest_claim_hcp_npi = pdlch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref1
    ON lch.latest_claim_hcp_npi = ref1.hcp_npi

-- Latest tx HCP + enrichment (provider + HCO)
LEFT JOIN most_recent_tx_hcp_final mrt
    ON ep.PATIENT_ID = mrt.PATIENT_ID
-- LEFT JOIN latest_treatment_hcp_visit_count lthvc
--     ON ep.PATIENT_ID = lthvc.PATIENT_ID
--    AND mrt.latest_treatment_hcp_npi = lthvc.latest_treatment_hcp_npi
LEFT JOIN provider_dim pdtch
    ON mrt.latest_treatment_hcp_npi = pdtch.npi
LEFT JOIN cmpa_insights_internal_schema.reference_file ref2
    ON mrt.latest_treatment_hcp_npi = ref2.hcp_npi

-- First Dx/Tx HCP attribution + stats
LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID
LEFT JOIN first_dx_hcp_5yr_stats fdhs
    ON ep.PATIENT_ID = fdhs.PATIENT_ID
LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_stats fths
    ON ep.PATIENT_ID = fths.PATIENT_ID
LEFT JOIN first_tx_hcp_5yr_tx_only fthtx
    ON ep.PATIENT_ID = fthtx.PATIENT_ID

-- Pivoted top-5 most-seen HCPs
LEFT JOIN most_seen_pivot msp
    ON ep.PATIENT_ID = msp.PATIENT_ID

ORDER BY ep.PATIENT_ID;

-- Materialize the temp view into the persistent base table.
CREATE OR Replace TEMPORARY VIEW patient360_base AS
SELECT DISTINCT * FROM patient_hcp_visit_summary;


In [0]:
%sql
-- =============================================================================
-- mpsii_tx_claims (Temp View)
--
-- What this view is:
--   A curated “treatment claims” (Tx) universe for an MPS II eligible patient cohort.
--   It outputs Tx events (medical NDC, pharmacy NDC, and procedure administrations)
--   with an attributed HCP NPI when available, within the refresh window.
--
-- Output grain:
--   One row per (patient_id, npi-attribution, fill_date) treatment event.
--   Note: NPI can be NULL (kept intentionally).
--
-- Key inputs:
--   - com_edp_prd.com_raw.kom_medical_events
--   - com_edp_prd.com_raw.kom_pharmacy_events
--   - com_raw.kom_providers  (for provider filtering)
--
-- Key logic:
--   1) Build eligible_patients using Dx evidence (E761/E763) + treatment evidence.
--   2) Define provider inclusion list (cohort_3_learnings) based on specialties.
--   3) Pull treatment events in refresh window and filter to included providers (or NULL NPI).
--
-- Parameters:
--   ${end_date} should be supplied by the runtime (Databricks widget/job parameter).
-- =============================================================================

CREATE OR REPLACE TEMP VIEW mpsii_tx_claims AS
WITH
-- ============================================================================
-- 1) Dx evidence for cohort building (Specified vs Unspecified)
--    These CTEs create Dx event streams used ONLY to count distinct Dx dates.
--    Window used here: 2020-08-01 → ${end_date}
-- ============================================================================

-- Specified Dx events:
--   Pull E761 diagnosis occurrences from:
--     (a) medical_events where DIAGNOSIS_CODES contains E761, using SERVICE_DATE
--     (b) pharmacy_events where DIAGNOSIS_CODE = E761 and transaction is PAID, using FILL_DATE
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Specified Dx patients:
--   Keep patients with at least 2 distinct Dx dates (>=2 distinct fill_date values).
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Unspecified Dx events:
--   Pull E763 diagnosis occurrences from medical + paid pharmacy, same as above.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Unspecified Dx patients:
--   Keep patients with at least 2 distinct Dx dates.
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- ============================================================================
-- 2) Treatment evidence for cohort building (refresh window only)
--    These CTEs do NOT output the final Tx universe; they’re used to confirm
--    that a patient has qualifying treatment evidence in the refresh window.
--    Window used here: 2023-08-01 → ${end_date}
-- ============================================================================

-- Treatment evidence (broad):
--   Patient qualifies if they have ANY of:
--     - Tivi NDCs in medical (NDC11) within refresh window
--     - Tivi NDCs in pharmacy within refresh window with PAID result
--     - Any procedure/admin codes in the provided procedure list within refresh window
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Treatment evidence (Tivi-only):
--   Narrower evidence set for incremental unspecified cohort:
--     - Tivi NDCs (medical/pharmacy) in refresh window
--     - J1743 procedure in refresh window
MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE = 'J1743'
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- ============================================================================
-- 3) Build eligible patient cohort
--    - Specified: >=2 E761 Dx dates AND ANY treatment evidence in refresh window
--    - Incremental unspecified: >=2 E763 Dx dates AND Tivi-only evidence
--      AND not already in specified+treatment cohort
-- ============================================================================

-- Specified cohort:
--   Patients who meet the “>=2 specified Dx dates” requirement
--   AND have at least one qualifying treatment event in refresh window.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Incremental unspecified cohort:
--   Patients who meet the “>=2 unspecified Dx dates” requirement
--   AND have Tivi-only evidence in refresh window
--   AND are not in the specified+treatment cohort (avoids double-counting).
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible cohort:
--   Union of specified+treatment cohort and incremental unspecified cohort.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ============================================================================
-- 4) Provider inclusion filter (cohort_3_learnings)
--    Goal: Restrict attributed NPIs to INDIVIDUAL providers that meet specialty rules.
--    This filter will be applied AFTER pulling treatment events.
--    Note: rows with NULL NPI are retained to preserve treatment dates even without attribution.
-- ============================================================================

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      -- Exclude a set of primary specialties considered out-of-scope/noise
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant',
        'Anesthesiology',
        'Dentist',
        'Dietitian, Registered',
        'Emergency Medical Technician, Basic',
        'Emergency Medicine',
        'General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered',
        'Obstetrics & Gynecology',
        'Pathology',
        'Radiology',
        'Urology'
      )
      -- OR explicitly include certain secondary specialties that are relevant
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry',
        'Psychiatry',
        'Adolescent Medicine',
        'Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine',
        'Nutrition, Pediatric',
        'Oncology, Pediatrics',
        'Pediatric Cardiology',
        'Pediatric Critical Care Medicine',
        'Pediatric Dermatology',
        'Pediatric Emergency Medicine',
        'Pediatric Endocrinology',
        'Pediatric Gastroenterology',
        'Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases',
        'Pediatric Nephrology',
        'Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery',
        'Pediatric Otolaryngology',
        'Pediatric Pulmonology',
        'Pediatric Radiology',
        'Pediatric Rehabilitation Medicine',
        'Pediatric Rheumatology',
        'Pediatric Surgery',
        'Pediatrics',
        'Clinical Biochemical Genetics',
        'Clinical Genetics (M.D.)',
        'Clinical Molecular Genetics',
        'Ph.D. Medical Genetics',
        'Neurodevelopmental Disabilities',
        'Neurology',
        'Neurology with Special Qualifications in Child Neurology',
        'Neuroradiology'
      )
    )
),

-- ============================================================================
-- 5) Treatment claims universe returned by the view (refresh/“2y” window)
--    Pull Tx events for eligible patients during 2023-08-01 → ${end_date}.
--    Sources and attribution rules:
--      A) Medical NDC events: NPI = COALESCE(rendering_npi, referring_npi), date = SERVICE_DATE
--      B) Pharmacy NDC events: NPI = prescriber_npi, date = FILL_DATE, PAID only
--      C) Procedure events:    NPI = rendering_npi, date = SERVICE_DATE
--    Then apply provider filter:
--      - keep if NPI in cohort_3_learnings OR NPI is NULL
-- ============================================================================

all_tx_claims_2yr AS (
    SELECT DISTINCT *
    FROM (
      -- Medical: Tivi NDC events with attributed HCP (rendering/referring)
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      -- Pharmacy: Tivi NDC fills with attributed prescriber (paid only)
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      -- UNION

      -- -- Medical: procedure/admin events with attributed renderer
      -- SELECT DISTINCT
      --   PATIENT_ID,
      --   RENDERING_NPI AS NPI,
      --   SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    -- Provider specialty filter: keep included providers OR keep NULL NPI rows
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
)

-- Final output: all treatment events in refresh window for the eligible cohort
SELECT * FROM all_tx_claims_2yr;


In [0]:
%sql
-- =============================================================================
-- most_recently_treated_hcp (Temp View)
--
-- What this view is:
--   Patient-level attribution of the “most recently treating HCP” within the
--   REFRESH treatment window (2023-08-01 → ${end_date}), plus HCP enrichment and
--   visit context metrics computed over a broader (5y-ish) claims universe.
--
-- Output grain:
--   One row per patient (only patients with an attributable treatment NPI in the
--   refresh window will appear, because latest_treating_hcp filters npi IS NOT NULL).
--
-- Key concepts:
--   - Selection window ("most recently treated"): tx_claims (refresh window)
--   - Context window ("visits / last seen"): all_claims (currently 2020-08-01 → ${end_date})
--   - Provider filter (cohort_3_learnings): keeps included INDIVIDUAL NPIs; retains NULL NPI rows
--     in claims universes, but final attribution requires non-null NPI.
--
-- Parameters:
--   ${end_date} must be provided by the runtime.
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW most_recently_treated_hcp AS
WITH
-- ============================================================================
-- 1) Treatment claims: refresh window source
--    This CTE simply points to the already-built tx universe from mpsii_tx_claims:
--      - eligible cohort already applied
--      - window already applied (2023-08-01 → ${end_date})
--      - provider filter already applied (allowed NPIs + NULL NPIs)
-- ============================================================================
tx_claims AS (
    SELECT DISTINCT *
    FROM mpsii_tx_claims
),

-- ============================================================================
-- 2) Re-derive eligible_patients cohort (duplicated here)
--    This block repeats the cohort logic so that the subsequent 5y Dx/Tx universes
--    (all_dx_claims_5yr / all_tx_claims_5yr) can be built inside this view.
--    NOTE: This duplication is intentional in the current script; no logic is changed.
-- ============================================================================

-- Specified Dx event stream (E761) within 2020-08-01 → ${end_date}
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Specified Dx patients: require ≥2 distinct Dx dates
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Unspecified Dx event stream (E763) within 2020-08-01 → ${end_date}
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Unspecified Dx patients: require ≥2 distinct Dx dates
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Qualifying treatment evidence (broad) in refresh window, used to ensure “active treatment”
MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
        --                          '38206','38230','38232','38240','38241','38242','38243','38250')
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Tivi-only evidence in refresh window (used only for incremental unspecified cohort)
MPSII_Treatment_Tivi_Only AS (
    SELECT DISTINCT PATIENT_ID FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('8497600101')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        UNION ALL
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('8497600101')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        -- UNION ALL
        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE = 'J1743'
        --   AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

-- Specified eligible: ≥2 specified Dx dates AND any qualifying treatment in refresh window
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Incremental unspecified eligible: ≥2 unspecified Dx dates AND Tivi-only tx in refresh window,
-- excluding already eligible specified+treatment patients
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Tivi_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible cohort used downstream for 5y Dx/Tx universes
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
),

-- ============================================================================
-- 3) Provider filter (cohort_3_learnings)
--    Defines an allowed set of INDIVIDUAL NPIs based on specialty inclusion rules.
--    Applied to all_dx_claims_5yr / all_tx_claims_5yr, with NULL NPIs retained.
-- ============================================================================
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- ============================================================================
-- 4) Build 5y-ish claims universes for visit counts / last-visit context
--    These are bounded by 2020-08-01 → ${end_date} and filtered to eligible_patients.
--    Provider filter is applied (allowed NPIs + NULL NPIs retained).
-- ============================================================================

-- Dx claims in 5y-ish window (E761/E763)
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Tx claims in 5y-ish window (Tivi NDCs + procedure list)
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('8497600101')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('8497600101')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      -- UNION
      -- SELECT DISTINCT
      --   PATIENT_ID,
      --   RENDERING_NPI AS NPI,
      --   SERVICE_DATE AS FILL_DATE
      -- FROM com_edp_prd.com_raw.kom_medical_events
      -- WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
      --                          '38206','38230','38232','38240','38241','38242','38243','38250')
      --   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
      --   AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Combined claims universe used only for:
--   - counting distinct visit dates per patient↔HCP
--   - computing last observed visit date per patient↔HCP
all_claims AS (
    SELECT * FROM all_dx_claims_5yr
    UNION
    SELECT * FROM all_tx_claims_5yr
),

-- ============================================================================
-- 5) Select the most recently treating HCP (refresh window)
--    Uses tx_claims (2023-08-01 → ${end_date}) to pick 1 NPI per patient:
--      - Prefer rows with non-null NPI
--      - Then pick the latest fill_date
--      - Tie-break by npi DESC (deterministic tie-breaker)
--    Final output from this CTE requires npi IS NOT NULL (attributable HCP).
-- ============================================================================
latest_treating_hcp AS (
    SELECT patient_id, npi
    FROM (
        SELECT
            patient_id,
            npi,
            fill_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY
                    CASE WHEN npi IS NOT NULL THEN 1 ELSE 2 END,  -- prioritize attributed claims
                    fill_date DESC,                               -- most recent date wins
                    npi DESC                                      -- deterministic tie-break
            ) AS rn
        FROM tx_claims
    ) t
    WHERE rn = 1
      AND npi IS NOT NULL
),

-- ============================================================================
-- 6) Compute visit metrics for the selected patient↔HCP pair (5y-ish universe)
--    These metrics are NOT limited to the refresh window; they use all_claims.
-- ============================================================================

-- Count of distinct visit dates for each patient↔HCP across all_claims
visit_counts AS (
    SELECT
        patient_id,
        npi,
        COUNT(DISTINCT fill_date) AS visit_counts
    FROM all_claims
    WHERE npi IS NOT NULL
    GROUP BY patient_id, npi
),

-- Last observed visit date for each selected patient↔HCP across all_claims
last_visit_date AS (
    SELECT
        lth.patient_id,
        lth.npi,
        MAX(ac.fill_date) AS last_visit_date
    FROM latest_treating_hcp lth
    LEFT JOIN all_claims ac
      ON lth.patient_id = ac.patient_id
     AND lth.npi = ac.npi
    GROUP BY lth.patient_id, lth.npi
),

-- Attach visit metrics to the most recently treated HCP per patient
latest_treating_hcp_with_visits AS (
    SELECT
        a.patient_id,
        a.npi AS most_recently_treated_hcp,
        b.visit_counts AS no_of_visits,
        c.last_visit_date AS last_visit_date_5yr
    FROM latest_treating_hcp AS a
    LEFT JOIN visit_counts AS b
        ON a.patient_id = b.patient_id
       AND a.npi        = b.npi
    LEFT JOIN last_visit_date AS c
        ON a.patient_id = c.patient_id
       AND a.npi        = c.npi
),

-- ============================================================================
-- 7) Enrichment: provider name/specialty + HCO + territory/region mapping
--    - provider info from kom_providers (INDIVIDUAL)
--    - HCO / territory / region crosswalk from reference_file
--      with additional territory_id/region_id derived via zip_to_territory_mapping
-- ============================================================================
hcp_with_other_info AS (
    SELECT
        a.*,
        CONCAT(b.FIRST_NAME, ' ', b.LAST_NAME) AS hcp_name,
        b.PRIMARY_SPECIALTY AS hcp_specialty,
        -- c.hco_npi,
        c.hco_name,
        c.mapped_territory_id AS territory_id,
        c.territory,
        c.mapped_region_id AS region_id,
        c.region
    FROM latest_treating_hcp_with_visits AS a

    -- Provider name and specialty enrichment (limit to INDIVIDUAL provider rows)
    LEFT JOIN com_edp_prd.com_raw.kom_providers AS b
        ON a.most_recently_treated_hcp = b.NPI
       AND b.PROVIDER_TYPE = 'INDIVIDUAL'

    -- Crosswalk HCP -> HCO and attach territory/region metadata.
    -- Inner derived tables map territory_name/region_name to numeric IDs.
    LEFT JOIN (
  SELECT
    * EXCEPT (hcp_primary_specialty),
    hcp_primary_specialty AS hcp_specialty
  FROM (
    SELECT
      a.* EXCEPT (territory_id, region_id),
      b.territory_id AS mapped_territory_id,
      c.region_id AS mapped_region_id
    FROM cmpa_insights_internal_schema.reference_file AS a
    LEFT JOIN (
      SELECT DISTINCT territory_id, territory_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
    ) AS b
      ON a.territory = b.territory_name
    LEFT JOIN (
      SELECT DISTINCT region_id, region_name
      FROM cmpa_insights_internal_schema.zip_to_territory_mapping
    ) AS c
      ON a.region = c.region_name
  )
) AS c
ON a.most_recently_treated_hcp = c.hcp_npi
)

-- ============================================================================
-- FINAL SELECT
--   Renames fields to make explicit:
--     - selection window: “_2yr” (refresh window)
--     - context metrics: “_5yr” (computed from all_claims 2020-08-01 → ${end_date})
-- ============================================================================
SELECT
    patient_id,
    most_recently_treated_hcp AS most_recently_treated_hcp_2yr,
    hcp_name AS most_recently_treated_hcp_name_2yr,

    -- Visit metrics are computed across all_claims (currently 2020-08-01 → ${end_date})
    no_of_visits AS most_recently_treated_hcp_2yr_no_of_visits_5yr,
    last_visit_date_5yr AS most_recent_tx_hcp_2yr_last_visit_5yr,

    hcp_specialty AS most_recently_treated_hcp_specialty_2yr,
    -- hco_npi AS most_recently_treated_hcp_hco_npi,
    hco_name AS most_recently_treated_hcp_hco_name,
    territory_id AS most_recently_treated_hcp_territory_id_2yr,
    territory AS most_recently_treated_hcp_territory_2yr,
    region_id AS most_recently_treated_hcp_region_id_2yr,
    region AS most_recently_treated_hcp_region_2yr
FROM hcp_with_other_info;


In [0]:
%sql
-- =============================================================================
-- Primary HCP Assignment (Dx + Tx Claims)
--
-- Goal
--   Assign ONE “primary HCP” (NPI) per eligible MPS II patient by ranking HCPs using:
--     Tier 1) Specialty priority
--     Tier 2) Total distinct visit dates (Dx + Tx combined)
--     Tier 3) Most recent visit date
--     Tier 4) NPI tiebreaker
--
-- Date windows used in this script
--   • Dx claim extraction (“5y” universe): 2020-08-01 → ${end_date}
--   • Tx claim extraction (“5y” universe): 2020-08-01 → ${end_date}
--   • Tx eligibility window (“2y/refresh”): 2023-08-01 → ${end_date}
--
-- NPI attribution (aligned to GTM file)
--   • Medical NDC claims:      COALESCE(RENDERING_NPI, REFERRING_NPI)
--   • Medical procedure claims:RENDERING_NPI
--   • Pharmacy claims:         PRESCRIBER_NPI
--
-- Eligibility overview (eligible_patients)
--   • Specified cohort:
--       - ≥2 distinct E761 Dx dates (medical or paid pharmacy) in Dx window
--       - AND any Tx in the 2y/refresh window
--   • Incremental unspecified cohort:
--       - ≥2 distinct E763 Dx dates (medical or paid pharmacy) in Dx window
--       - AND Tivi-coded Tx in the 2y/refresh window (NDCs or J1743)
--       - Excludes patients already in specified cohort
-- =============================================================================


-- =============================================================================
-- STEP 1: DIAGNOSIS CLAIMS (Dx universe for visit counting)
--   • Captures E761/E763 diagnosis evidence from medical + paid pharmacy events
--   • Window: 2020-08-01 → ${end_date}
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

-- Medical events Dx (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Pharmacy events Dx (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'DX' AS CLAIM_TYPE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 2: TREATMENT CLAIMS (Tx universe for visit counting)
--   • Captures Tivi-coded treatment from medical NDC, medical procedures, and paid pharmacy NDC
--   • Window: 2020-08-01 → ${end_date}
--   • Includes CODE field to retain NDC/procedure provenance
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

-- Medical events Tx via NDC (NPI = COALESCE(rendering, referring))
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('8497600101')
  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

-- UNION

-- -- Medical events Tx via procedures (NPI = rendering_npi only)
-- SELECT DISTINCT
--     PATIENT_ID,
--     RENDERING_NPI AS NPI,
--     SERVICE_DATE AS FILL_DATE,
--     'TX' AS CLAIM_TYPE,
--     PROCEDURE_CODE AS CODE
-- FROM com_edp_prd.com_raw.kom_medical_events
-- WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
--                          'S9357', 'S9379', '38206', '38230', '38232',
--                          '38240', '38241', '38242', '38243', '38250')
--   AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION

-- Pharmacy events Tx via NDC (NPI = prescriber_npi; paid only)
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    'TX' AS CLAIM_TYPE,
    NDC11 AS CODE
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('8497600101')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 3: Tx CLAIMS IN ELIGIBILITY WINDOW (2y/refresh)
--   • Subset of all_tx_claims restricted to 2023-08-01 → ${end_date}
--   • Used ONLY to determine cohort eligibility (not for visit counting tiers)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW tx_claims_2yr AS
SELECT DISTINCT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE, CODE
FROM all_tx_claims
WHERE FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters);


-- =============================================================================
-- STEP 4: PATIENT ELIGIBILITY
--   Builds eligible_patients using Dx evidence (≥2 dates) + Tx evidence in 2y window.
-- =============================================================================

-- 4A) Specified Dx requirement: ≥2 distinct E761 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4B) Specified cohort: specified Dx + any Tx in 2y/refresh window
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN tx_claims_2yr t ON e.PATIENT_ID = t.PATIENT_ID;

-- 4C) Incremental Dx requirement: ≥2 distinct E763 Dx dates in Dx window
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;

-- 4D) Tivi-coded Tx requirement for incremental eligibility (2y/refresh window)
CREATE OR REPLACE TEMPORARY VIEW Tivi_tx_2yr AS
SELECT DISTINCT PATIENT_ID
FROM tx_claims_2yr
WHERE CODE IN ('8497600101');

-- 4E) Incremental cohort: incremental Dx + Tivi-coded tx in 2y window + exclude specified
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN Tivi_tx_2yr t ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM specified_patients);

-- 4F) Final eligible cohort
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;


-- =============================================================================
-- Provider inclusion list (INDIVIDUAL NPIs only)
--   • Used to restrict HCPs considered for primary assignment
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW cohort_3_learnings AS
SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
    PRIMARY_SPECIALTY NOT IN (
      'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
      'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
      'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
      'Radiology','Urology'
    )
    OR SECONDARY_SPECIALTY IN (
      'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
      'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
      'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
      'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
      'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
      'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
      'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
      'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
      'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
      'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
    )
  );


-- =============================================================================
-- STEP 5: COMBINED Dx + Tx CLAIMS FOR ELIGIBLE PATIENTS
--   • Combines Dx + Tx events (visit dates) for eligible patients only
--   • Filters to included INDIVIDUAL NPIs (cohort_3_learnings)
--   • NOTE: As written, this excludes NULL NPI rows (because of the IN filter)
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS
SELECT *
FROM (
  -- Dx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_dx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

  UNION

  -- Tx claims
  SELECT PATIENT_ID, NPI, FILL_DATE, CLAIM_TYPE
  FROM all_tx_claims
  WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
)
WHERE npi IN (SELECT DISTINCT npi FROM cohort_3_learnings);


-- =============================================================================
-- STEP 6: PRIMARY HCP ASSIGNMENT (4-tier ranking)
--   Tier 1: Specialty priority bucket (lower = better)
--   Tier 2: Total distinct visit dates (Dx + Tx)
--   Tier 3: Most recent visit date
--   Tier 4: NPI tiebreaker
-- =============================================================================
CREATE OR REPLACE TEMPORARY VIEW primary_hcp AS
WITH hcp_metrics AS (
    SELECT
        a.PATIENT_ID,
        a.NPI,

        -- Specialty bucket (reporting label)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'
            WHEN a.NPI IS NULL
                THEN 'NA'
            ELSE 'Others'
        END AS SPECIALTY,

        -- Tier 1: specialty priority (lower is better)
        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1
            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2
            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3
            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4
            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5
            WHEN a.NPI IS NULL
                THEN 7
            ELSE 6
        END AS SPECIALTY_PRIORITY,

        -- Tier 2: visit counts (Dx + Tx combined)
        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        -- Reference-only breakdowns (not used in ranking)
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,
        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        -- Tier 3: most recent visit date
        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims a
    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI
    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),
ranked_hcps AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK
    FROM hcp_metrics
)
SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;

-- Optional materialization:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp AS
-- SELECT * FROM primary_hcp;


In [0]:
-- %sql
-- -- =============================================================================
-- -- STEP 7: Materialize Primary HCP table with HCP identity + HCO/territory metadata
-- --
-- -- What this step does:
-- --   Takes the existing primary_hcp assignment (already computed upstream)
-- --   and persists it as a physical table, while enriching it with:
-- --     - HCP full name (from kom_providers)
-- --     - HCO affiliation + territory/region attributes (from reference_file)
-- --     - territory_id / region_id (derived via zip_to_territory_mapping lookups)
-- --
-- -- What this step does NOT do:
-- --   - It does not re-rank or change the primary HCP selection logic.
-- --   - It does not filter the cohort; it simply enriches and stores primary_hcp rows.
-- --
-- -- Output:
-- --   com_edp_prd.cmpa_insights_internal_schema.primary_hcp
-- -- =============================================================================

-- CREATE OR REPLACE TEMPORARY VIEW primary_hcp_base AS
-- SELECT
--     ph.*,  -- retain all columns produced by the upstream primary_hcp view/table

--     -- -------------------------------------------------------------------------
--     -- HCP display name enrichment
--     --   Concatenate first + last name from provider dimension; COALESCE protects
--     --   against nulls so string concat doesn't produce NULL.
--     -- -------------------------------------------------------------------------
--     COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '') AS primary_hcp_name_2yr,

--     -- -------------------------------------------------------------------------
--     -- HCO affiliation enrichment (via crosswalk)
--     --   Map HCP NPI -> HCO NPI / HCO name using reference_file.
--     -- -------------------------------------------------------------------------
--     -- ref.HCO_NPI  AS primary_hcp_hco_npi_2yr,
--     ref.HCO_NAME AS primary_hcp_hco_name_2yr,

--     -- -------------------------------------------------------------------------
--     -- Territory / region enrichment
--     --   Pull both the human-readable names and numeric IDs (territory_id, region_id).
--     -- -------------------------------------------------------------------------
--     ref.mapped_territory_id AS primary_hcp_territory_id_2yr,
--     ref.TERRITORY           AS primary_hcp_territory_2yr,
--     ref.mapped_region_id    AS primary_hcp_region_id_2yr,
--     ref.region              AS primary_hcp_region_2yr
-- FROM primary_hcp ph

-- -- -----------------------------------------------------------------------------
-- -- Join #1: Provider dimension (name enrichment)
-- --   Join on the attributed primary HCP NPI to retrieve FIRST_NAME / LAST_NAME.
-- -- -----------------------------------------------------------------------------
-- LEFT JOIN com_edp_prd.com_raw.kom_providers p
--     ON ph.PRIMARY_HCP_NPI = p.NPI

-- -- -----------------------------------------------------------------------------
-- -- Join #2: Reference enrichment (HCO + territory + region)
-- --   Build a reference subquery that:
-- --     1) Starts from reference_file (HCP ↔ HCO + territory/region names)
-- --     2) Adds territory_id by mapping territory name -> territory_id via zip_to_territory_mapping
-- --     3) Adds region_id by mapping region name -> region_id via zip_to_territory_mapping
-- --     4) Renames hcp_primary_specialty to hcp_specialty (not used in final select here,
-- --        but retained in the ref dataset)
-- --   Finally join ref to primary_hcp using HCP_NPI.
-- -- -----------------------------------------------------------------------------
-- LEFT JOIN (
--     SELECT
--       * EXCEPT (hcp_primary_specialty),
--       hcp_primary_specialty AS hcp_specialty
--     FROM (
--       SELECT
--   a.* EXCEPT (territory_id, region_id),   -- remove duplicates
--   b.territory_id AS mapped_territory_id,
--   c.region_id AS mapped_region_id
--       FROM cmpa_insights_internal_schema.reference_file AS a

--       -- Map territory name -> territory_id (distinct pairs)
--       LEFT JOIN (
--         SELECT DISTINCT territory_id, territory_name
--         FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--       ) AS b
--         ON a.territory = b.territory_name

--       -- Map region name -> region_id (distinct pairs)
--       LEFT JOIN (
--         SELECT DISTINCT region_id, region_name
--         FROM cmpa_insights_internal_schema.zip_to_territory_mapping
--       ) AS c
--         ON a.region = c.region_name
--     )
-- ) ref
--     ON ph.PRIMARY_HCP_NPI = ref.HCP_NPI;


In [0]:
%sql
-- =============================================================================
-- patient360_master (Final patient-level master table)
--
-- What this step does:
--   Materializes a single, “wide” patient-level table by combining three upstream
--   datasets into one row per patient:
--     A) patient360_base              -> core demographics + claim-derived milestones + HCP features
--     B) most_recently_treated_hcp    -> most recent treating HCP in refresh window + visit context metrics
--     C) primary_hcp                  -> primary HCP assignment + HCO/territory/region enrichment
--
-- Output:
--   com_edp_prd.cmpa_insights_internal_schema.patient360_master
--
-- Join strategy:
--   - Start from patient360_base (a) as the backbone (all patients retained)
--   - LEFT JOIN most_recently_treated_hcp (b) on patient_id to add recent-treatment attribution fields
--   - LEFT JOIN primary_hcp (c) on patient_id to add primary HCP and territory enrichment fields
--   - SELECT DISTINCT used to dedupe in case joins introduce multiplicity (e.g., enrichment tables
--     or upstream views have >1 row per patient)
--
-- Column handling:
--   - b.* EXCEPT(patient_id) and c.* EXCEPT(patient_id) avoid duplicating patient_id columns
--     from the joined datasets.
--   - Window suffixes (_2yr/_3yr/_5yr) indicate derivation windows from upstream logic.
--
-- Parameters:
--   None directly here (all parameterized logic occurs upstream), but this depends on upstream
--   tables/views that may be parameterized by ${end_date}.
-- =============================================================================

CREATE OR REPLACE TEMPORARY VIEW tivi_patient360_master AS
select PATIENT_ID as tivi_patient_id, PATIENT_YOB as tivi_patient_yob, PATIENT_AGE as tivi_patient_age, PATIENT_GENDER as tivi_patient_gender, patient_state as tivi_patient_state, incidence_date as tivi_incidence_date, first_incidence_treatment_date as tivi_first_incidence_treatment_date, latest_claim_date as tivi_latest_claim_date, latest_claim_hcp_npi as tivi_latest_claim_hcp_npi, latest_claim_hcp_name as tivi_latest_claim_hcp_name, latest_claim_hcp_specialty as tivi_latest_claim_hcp_specialty, latest_claim_hcp_visit_count as tivi_latest_claim_hcp_visit_count, latest_claim_hcp_hco_name as tivi_latest_claim_hcp_hco_name, latest_treatment_date as tivi_latest_treatment_date, latest_mpsii_tx_type as tivi_latest_mpsii_tx_type, first_tx_after_diagnosis as tivi_first_tx_after_diagnosis, time_dx_to_first_tx_in_months as tivi_time_dx_to_first_tx_in_months, treatment_period_months as tivi_treatment_period_months, Tivi_fills as tivi_fills, latest_treatment_hcp_npi as tivi_latest_treatment_hcp_npi, latest_treatment_hcp_name as tivi_latest_treatment_hcp_name, latest_treatment_hcp_specialty as tivi_latest_treatment_hcp_specialty, latest_treatment_hcp_visit_count as tivi_latest_treatment_hcp_visit_count, latest_treatment_hcp_hco_name as tivi_latest_treatment_hcp_hco_name, first_dx_hcp_5yr as tivi_first_dx_hcp_5yr, first_dx_all_visit_count_5yr as tivi_first_dx_all_visit_count_5yr, first_dx_last_visit_5yr as tivi_first_dx_last_visit_5yr, first_tx_hcp_5yr as tivi_first_tx_hcp_5yr, first_tx_all_visit_count_5yr as tivi_first_tx_all_visit_count_5yr, first_tx_treatment_visit_count_5yr as tivi_first_tx_treatment_visit_count_5yr, first_tx_last_visit_5yr as tivi_first_tx_last_visit_5yr, most_seen_hcp1_3yr_ranked as tivi_most_seen_hcp1_3yr_ranked, most_seen_hcp1_visit_count_5yr as tivi_most_seen_hcp1_visit_count_5yr, most_seen_hcp1_last_visit_5yr as tivi_most_seen_hcp1_last_visit_5yr, most_seen_hcp2_3yr_ranked as tivi_most_seen_hcp2_3yr_ranked, most_seen_hcp2_visit_count_5yr as tivi_most_seen_hcp2_visit_count_5yr, most_seen_hcp2_last_visit_5yr as tivi_most_seen_hcp2_last_visit_5yr, most_seen_hcp3_3yr_ranked as tivi_most_seen_hcp3_3yr_ranked, most_seen_hcp3_visit_count_5yr as tivi_most_seen_hcp3_visit_count_5yr, most_seen_hcp3_last_visit_5yr as tivi_most_seen_hcp3_last_visit_5yr, most_seen_hcp4_3yr_ranked as tivi_most_seen_hcp4_3yr_ranked, most_seen_hcp4_visit_count_5yr as tivi_most_seen_hcp4_visit_count_5yr, most_seen_hcp4_last_visit_5yr as tivi_most_seen_hcp4_last_visit_5yr, most_seen_hcp5_3yr_ranked as tivi_most_seen_hcp5_3yr_ranked, most_seen_hcp5_visit_count_5yr as tivi_most_seen_hcp5_visit_count_5yr, most_seen_hcp5_last_visit_5yr as tivi_most_seen_hcp5_last_visit_5yr, most_recently_treated_hcp_2yr as tivi_most_recently_treated_hcp_2yr, most_recently_treated_hcp_name_2yr as tivi_most_recently_treated_hcp_name_2yr, most_recently_treated_hcp_2yr_no_of_visits_5yr as tivi_most_recently_treated_hcp_2yr_no_of_visits_5yr, most_recent_tx_hcp_2yr_last_visit_5yr as tivi_most_recent_tx_hcp_2yr_last_visit_5yr, most_recently_treated_hcp_specialty_2yr as tivi_most_recently_treated_hcp_specialty_2yr, most_recently_treated_hcp_hco_name as tivi_most_recently_treated_hcp_hco_name, most_recently_treated_hcp_territory_id_2yr as tivi_most_recently_treated_hcp_territory_id_2yr, most_recently_treated_hcp_territory_2yr as tivi_most_recently_treated_hcp_territory_2yr, most_recently_treated_hcp_region_id_2yr as tivi_most_recently_treated_hcp_region_id_2yr, most_recently_treated_hcp_region_2yr as tivi_most_recently_treated_hcp_region_2yr, PRIMARY_HCP_NPI as tivi_primary_hcp_npi, PRIMARY_HCP_SPECIALTY as tivi_primary_hcp_specialty, SPECIALTY_PRIORITY as tivi_specialty_priority, NO_OF_VISITS as tivi_no_of_visits, DX_VISITS as tivi_dx_visits, TX_VISITS as tivi_tx_visits, MOST_RECENT_VISIT as tivi_most_recent_visit, HCP_RANK as tivi_hcp_rank


from (SELECT DISTINCT
    -- -------------------------------------------------------------------------
    -- Patient identity + demographics (from patient360_base)
    -- -------------------------------------------------------------------------
    a.PATIENT_ID,
    a.PATIENT_YOB,
    a.PATIENT_AGE,
    a.PATIENT_GENDER,
    a.patient_state,

    -- -------------------------------------------------------------------------
    -- Historical milestones (patient-level; timeframe-agnostic where defined upstream)
    --   - incidence_date: earliest observed Dx date across history (as defined in base)
    --   - first_incidence_treatment_date: earliest observed treatment date across history (as defined in base)
    -- -------------------------------------------------------------------------
    a.incidence_date,
    a.first_incidence_treatment_date,

    -- -------------------------------------------------------------------------
    -- Latest claim (patient-level)
    --   - latest_claim_date may fall back to a patient-level date if NPI attribution is missing upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_date,

    -- -------------------------------------------------------------------------
    -- Latest claim attributed HCP + visit context (from patient360_base)
    --   Note: these fields are populated only if the latest claim had an attributable NPI upstream
    -- -------------------------------------------------------------------------
    a.latest_claim_hcp_npi,
    a.latest_claim_hcp_name,
    a.latest_claim_hcp_specialty,
    a.latest_claim_hcp_visit_count,

    -- Latest claim attributed HCO (via reference enrichment in base)
    -- a.latest_claim_hcp_hco_npi,
    a.latest_claim_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- Latest treatment (patient-level) + derived treatment type
    --   - latest_treatment_date may fall back to a patient-level date if NPI attribution is missing upstream
    --   - latest_mpsii_tx_type typically derived within the refresh window upstream
    -- -------------------------------------------------------------------------
    a.latest_treatment_date,
    a.latest_mpsii_tx_type,

    -- -------------------------------------------------------------------------
    -- First Tx after Dx + timelines (from patient360_base)
    --   - first_tx_after_diagnosis is computed upstream (all-time tx constrained to >= incidence_date)
    --   - timelines are derived from incidence_date / first_tx_after_diagnosis / latest_treatment_date
    -- -------------------------------------------------------------------------
    a.first_tx_after_diagnosis,
    a.time_dx_to_first_tx_in_months,
    a.treatment_period_months,

    -- -------------------------------------------------------------------------
    -- Treatment activity proxy (from patient360_base)
    --   - Tivi_fills: distinct tx dates in refresh window as defined upstream
    -- -------------------------------------------------------------------------
    a.Tivi_fills,

    -- -------------------------------------------------------------------------
    -- Latest treatment attributed HCP + visit context (from patient360_base)
    -- -------------------------------------------------------------------------
    a.latest_treatment_hcp_npi,
    a.latest_treatment_hcp_name,
    a.latest_treatment_hcp_specialty,
    a.latest_treatment_hcp_visit_count,

    -- Latest treatment attributed HCO (via reference enrichment in base)
    -- a.latest_treatment_hcp_hco_npi,
    a.latest_treatment_hcp_hco_name,

    -- -------------------------------------------------------------------------
    -- First Dx / Tx attributed HCP metrics (5y-ish universe; from patient360_base)
    -- -------------------------------------------------------------------------
    a.first_dx_hcp_5yr,
    a.first_dx_all_visit_count_5yr,
    a.first_dx_last_visit_5yr,
    a.first_tx_hcp_5yr,
    a.first_tx_all_visit_count_5yr,
    a.first_tx_treatment_visit_count_5yr,
    a.first_tx_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Top-5 most-seen HCPs (ranked by 3y; includes 5y counts + last-visit; from patient360_base)
    -- -------------------------------------------------------------------------
    a.most_seen_hcp1_3yr_ranked,
    a.most_seen_hcp1_visit_count_5yr,
    a.most_seen_hcp1_last_visit_5yr,
    a.most_seen_hcp2_3yr_ranked,
    a.most_seen_hcp2_visit_count_5yr,
    a.most_seen_hcp2_last_visit_5yr,
    a.most_seen_hcp3_3yr_ranked,
    a.most_seen_hcp3_visit_count_5yr,
    a.most_seen_hcp3_last_visit_5yr,
    a.most_seen_hcp4_3yr_ranked,
    a.most_seen_hcp4_visit_count_5yr,
    a.most_seen_hcp4_last_visit_5yr,
    a.most_seen_hcp5_3yr_ranked,
    a.most_seen_hcp5_visit_count_5yr,
    a.most_seen_hcp5_last_visit_5yr,

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #1: most_recently_treated_hcp
    --   Pulls in:
    --     - most_recently_treated_hcp_2yr and its attributes (name/specialty/HCO/territory/region)
    --     - visit context metrics computed over the broader claims universe in that view
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    b.* EXCEPT (patient_id),

    -- -------------------------------------------------------------------------
    -- Add-on enrichment block #2: primary_hcp
    --   Pulls in:
    --     - primary HCP attribution + name/HCO/territory/region fields
    --   EXCEPT(patient_id) avoids duplicating the patient id column.
    -- -------------------------------------------------------------------------
    c.* EXCEPT (patient_id)

FROM patient360_base AS a

-- Left join retains all patients from patient360_base even if no match in most_recently_treated_hcp
LEFT JOIN most_recently_treated_hcp AS b
    ON a.PATIENT_ID = b.patient_id

-- Left join retains all patients from patient360_base even if no match in primary_hcp
LEFT JOIN primary_hcp AS c
    ON a.PATIENT_ID = c.patient_id)


In [0]:
create or replace table com_edp_prd.cmpa_insights_internal_schema.patient360_master
with base as (
  select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
),
t2 as (select a.*, b.*except (b.tivi_patient_id, b.tivi_patient_yob, b.tivi_patient_age, b.tivi_patient_gender, b.tivi_patient_state)
from base as a
left join tivi_patient360_master as b on a.patient_id = b.tivi_patient_id)
select distinct * from t2

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master
limit 10;

### Testing

In [0]:
select count(distinct PATIENT_ID) as count_patient from com_edp_prd.cmpa_insights_internal_schema.patient360_master
where latest_treatment_hcp_name is null;

In [0]:
-- =============================================================================
-- Extract ALL Claims (Medical + Pharmacy) - EXACT COPY from patient360 logic
-- Purpose: Get all claims in raw format for manual parsing
-- =============================================================================

-- Note: Replace ${end_date} with your actual end date, e.g., '2025-12-31'

WITH

-- Same eligible patients logic from your code
eligible_patients AS (
    SELECT p.PATIENT_ID FROM (
        SELECT PATIENT_ID
        FROM (
            SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
            FROM com_edp_prd.com_raw.kom_medical_events
            WHERE DIAGNOSIS_CODES LIKE '%E761%'
              AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
            UNION
            SELECT DISTINCT PATIENT_ID, FILL_DATE
            FROM com_edp_prd.com_raw.kom_pharmacy_events
            WHERE DIAGNOSIS_CODE = 'E761'
              AND TRANSACTION_STATUS = 'PAID'
              AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        ) dx
        GROUP BY PATIENT_ID
        HAVING COUNT(DISTINCT FILL_DATE) >= 2
    ) p
    INNER JOIN (
        SELECT DISTINCT PATIENT_ID FROM (
            SELECT DISTINCT PATIENT_ID
            FROM com_edp_prd.com_raw.kom_medical_events
            WHERE NDC11 IN ('54092070001','540920700')
              AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
            UNION ALL
            SELECT DISTINCT PATIENT_ID
            FROM com_edp_prd.com_raw.kom_pharmacy_events
            WHERE NDC11 IN ('54092070001','540920700')
              AND TRANSACTION_RESULT = 'PAID'
              AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
            UNION ALL
            SELECT DISTINCT PATIENT_ID
            FROM com_edp_prd.com_raw.kom_medical_events
            WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                     '38206','38230','38232','38240','38241','38242','38243','38250')
              AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        ) t
    ) tx ON p.PATIENT_ID = tx.PATIENT_ID
    
    UNION
    
    SELECT p.PATIENT_ID FROM (
        SELECT PATIENT_ID
        FROM (
            SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
            FROM com_edp_prd.com_raw.kom_medical_events
            WHERE DIAGNOSIS_CODES LIKE '%E763%'
              AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
            UNION
            SELECT DISTINCT PATIENT_ID, FILL_DATE
            FROM com_edp_prd.com_raw.kom_pharmacy_events
            WHERE DIAGNOSIS_CODE = 'E763'
              AND TRANSACTION_STATUS = 'PAID'
              AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        ) dx
        GROUP BY PATIENT_ID
        HAVING COUNT(DISTINCT FILL_DATE) >= 2
    ) p
    INNER JOIN (
        SELECT DISTINCT PATIENT_ID FROM (
            SELECT DISTINCT PATIENT_ID
            FROM com_edp_prd.com_raw.kom_medical_events
            WHERE NDC11 IN ('54092070001','540920700')
              AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
            UNION ALL
            SELECT DISTINCT PATIENT_ID
            FROM com_edp_prd.com_raw.kom_pharmacy_events
            WHERE NDC11 IN ('54092070001','540920700')
              AND TRANSACTION_RESULT = 'PAID'
              AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
            UNION ALL
            SELECT DISTINCT PATIENT_ID
            FROM com_edp_prd.com_raw.kom_medical_events
            WHERE PROCEDURE_CODE = 'J1743'
              AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
        ) t
    ) tx ON p.PATIENT_ID = tx.PATIENT_ID
    WHERE p.PATIENT_ID NOT IN (
        SELECT PATIENT_ID FROM (
            SELECT PATIENT_ID
            FROM (
                SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
                FROM com_edp_prd.com_raw.kom_medical_events
                WHERE DIAGNOSIS_CODES LIKE '%E761%'
                  AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
                UNION
                SELECT DISTINCT PATIENT_ID, FILL_DATE
                FROM com_edp_prd.com_raw.kom_pharmacy_events
                WHERE DIAGNOSIS_CODE = 'E761'
                  AND TRANSACTION_STATUS = 'PAID'
                  AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
            ) dx
            GROUP BY PATIENT_ID
            HAVING COUNT(DISTINCT FILL_DATE) >= 2
        )
    )
),

-- Same cohort_3_learnings filter from your code
cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- EXACT COPY: all_dx_claims_5yr from your code
all_dx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT PATIENT_ID, COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI, SERVICE_DATE AS FILL_DATE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
      UNION
      SELECT DISTINCT PATIENT_ID, PRESCRIBER_NPI AS NPI, FILL_DATE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE DIAGNOSIS_CODE IN ('E761','E763')
        AND TRANSACTION_STATUS = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- EXACT COPY: all_tx_claims_5yr from your code
all_tx_claims_5yr AS (
    SELECT DISTINCT *
    FROM (
      SELECT DISTINCT
          PATIENT_ID,
          COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
          SERVICE_DATE AS FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          PRESCRIBER_NPI AS NPI,
          FILL_DATE,
          NDC11 AS TX_CODE
      FROM com_edp_prd.com_raw.kom_pharmacy_events
      WHERE NDC11 IN ('54092070001','540920700')
        AND TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

      UNION

      SELECT DISTINCT
          PATIENT_ID,
          RENDERING_NPI AS NPI,
          SERVICE_DATE AS FILL_DATE,
          PROCEDURE_CODE AS TX_CODE
      FROM com_edp_prd.com_raw.kom_medical_events
      WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                               '38206','38230','38232','38240','38241','38242','38243','38250')
        AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
        AND PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)
    )
    WHERE npi IN (SELECT npi FROM cohort_3_learnings) OR npi IS NULL
),

-- Combine Dx + Tx with labels
all_claims AS (
    SELECT 
        PATIENT_ID,
        NPI,
        FILL_DATE,
        'DIAGNOSIS' AS CLAIM_TYPE,
        CAST(NULL AS STRING) AS TX_CODE
    FROM all_dx_claims_5yr
    
    UNION ALL
    
    SELECT 
        PATIENT_ID,
        NPI,
        FILL_DATE,
        'TREATMENT' AS CLAIM_TYPE,
        TX_CODE
    FROM all_tx_claims_5yr
),

-- EXACT COPY: Get latest CLAIM (Dx or Tx) per patient (HCP-attributed, requires NPI)
-- This matches latest_claim_hcp_ranked from your code
latest_claim_hcp_ranked AS (
    SELECT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY FILL_DATE DESC, NPI ASC
        ) AS rn
    FROM all_claims
    WHERE NPI IS NOT NULL
),
latest_claim_hcp AS (
    SELECT
        PATIENT_ID,
        NPI AS latest_claim_hcp_npi,
        FILL_DATE AS latest_claim_date
    FROM latest_claim_hcp_ranked
    WHERE rn = 1
),

-- Patient-level fallback for latest claim date (includes NULL NPI claims)
latest_claim_date_patient AS (
    SELECT 
        PATIENT_ID,
        MAX(FILL_DATE) AS latest_claim_date_any
    FROM all_claims
    GROUP BY PATIENT_ID
),

-- EXACT COPY: Get latest TREATMENT/INFUSION claim per patient (HCP-attributed, requires NPI)
-- This matches most_recent_tx_hcp_ranked from your code
most_recent_tx_hcp_ranked AS (
    SELECT
        PATIENT_ID,
        NPI,
        FILL_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY FILL_DATE DESC, NPI ASC
        ) AS rn
    FROM all_claims
    WHERE CLAIM_TYPE = 'TREATMENT'
      AND NPI IS NOT NULL
),
most_recent_tx_hcp AS (
    SELECT
        PATIENT_ID,
        NPI AS latest_treatment_hcp_npi,
        FILL_DATE AS latest_treatment_date
    FROM most_recent_tx_hcp_ranked
    WHERE rn = 1
),

-- Patient-level fallback for latest treatment date (includes NULL NPI claims)
latest_treatment_date_patient AS (
    SELECT 
        PATIENT_ID,
        MAX(FILL_DATE) AS latest_treatment_date_any
    FROM all_claims
    WHERE CLAIM_TYPE = 'TREATMENT'
    GROUP BY PATIENT_ID
),

-- Get all unique patients
all_patients AS (
    SELECT DISTINCT PATIENT_ID
    FROM all_claims
)

-- Final output: patient_id | latest claim hcp | latest claim date | latest infusion hcp | latest infusion date
SELECT 
    p.PATIENT_ID AS patient_id,
    lch.latest_claim_hcp_npi AS latest_claim_hcp,
    COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
    mrt.latest_treatment_hcp_npi AS latest_infusion_hcp,
    COALESCE(mrt.latest_treatment_date, ltd.latest_treatment_date_any) AS latest_infusion_date
FROM all_patients p
LEFT JOIN latest_claim_hcp lch
    ON p.PATIENT_ID = lch.PATIENT_ID
LEFT JOIN latest_claim_date_patient lcd
    ON p.PATIENT_ID = lcd.PATIENT_ID
LEFT JOIN most_recent_tx_hcp mrt
    ON p.PATIENT_ID = mrt.PATIENT_ID
LEFT JOIN latest_treatment_date_patient ltd
    ON p.PATIENT_ID = ltd.PATIENT_ID
ORDER BY patient_id ASC;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.reference_file
where hcp_name ilike '%julie%';

In [0]:
-- =============================================================================
-- ALL CLAIMS for Territory 2003 Patients
-- =============================================================================

WITH territory_2003_patients AS (
    SELECT DISTINCT patient_id
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr = '2003'
),

-- Get all diagnosis claims
all_dx_claims AS (
    SELECT 
        t.patient_id,
        'MEDICAL_DX' AS claim_source,
        me.SERVICE_DATE AS claim_date,
        me.DIAGNOSIS_CODES,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi,
        me.RENDERING_NPI,
        me.REFERRING_NPI,
        CAST(NULL AS STRING) AS ndc11,
        CAST(NULL AS STRING) AS procedure_code
    FROM territory_2003_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE (me.DIAGNOSIS_CODES LIKE '%E761%' OR me.DIAGNOSIS_CODES LIKE '%E763%')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        'PHARMACY_DX' AS claim_source,
        pe.FILL_DATE AS claim_date,
        pe.DIAGNOSIS_CODE AS diagnosis_codes,
        pe.PRESCRIBER_NPI AS npi,
        CAST(NULL AS STRING) AS rendering_npi,
        CAST(NULL AS STRING) AS referring_npi,
        CAST(NULL AS STRING) AS ndc11,
        CAST(NULL AS STRING) AS procedure_code
    FROM territory_2003_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.DIAGNOSIS_CODE IN ('E761','E763')
      AND pe.TRANSACTION_STATUS = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
),

-- Get all treatment claims
all_tx_claims AS (
    SELECT 
        t.patient_id,
        'MEDICAL_NDC' AS claim_source,
        me.SERVICE_DATE AS claim_date,
        me.DIAGNOSIS_CODES,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi,
        me.RENDERING_NPI,
        me.REFERRING_NPI,
        me.NDC11,
        CAST(NULL AS STRING) AS procedure_code
    FROM territory_2003_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.NDC11 IN ('54092070001','540920700')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        'PHARMACY_NDC' AS claim_source,
        pe.FILL_DATE AS claim_date,
        pe.DIAGNOSIS_CODE AS diagnosis_codes,
        pe.PRESCRIBER_NPI AS npi,
        CAST(NULL AS STRING) AS rendering_npi,
        CAST(NULL AS STRING) AS referring_npi,
        pe.NDC11,
        CAST(NULL AS STRING) AS procedure_code
    FROM territory_2003_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.NDC11 IN ('54092070001','540920700')
      AND pe.TRANSACTION_RESULT = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        'MEDICAL_PROC' AS claim_source,
        me.SERVICE_DATE AS claim_date,
        me.DIAGNOSIS_CODES,
        me.RENDERING_NPI AS npi,
        me.RENDERING_NPI,
        CAST(NULL AS STRING) AS referring_npi,
        CAST(NULL AS STRING) AS ndc11,
        me.PROCEDURE_CODE
    FROM territory_2003_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                '38206','38230','38232','38240','38241','38242','38243','38250')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
),

-- Combine all claims
all_claims AS (
    SELECT * FROM all_dx_claims
    UNION ALL
    SELECT * FROM all_tx_claims
)

-- Final output
SELECT 
    patient_id,
    claim_source,
    claim_date,
    diagnosis_codes,
    npi,
    rendering_npi,
    referring_npi,
    ndc11,
    procedure_code
    
FROM all_claims
ORDER BY 
    patient_id ASC,
    claim_date DESC;

In [0]:
-- =============================================================================
-- ALL Territories: Patient-Level Latest Claim & Infusion Analysis
-- Same logic as territory 2003, but for all territories
-- =============================================================================

WITH all_territory_patients AS (
    SELECT DISTINCT 
        patient_id,
        primary_hcp_territory_id_2yr AS territory
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
),

-- Get all diagnosis claims
all_dx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE (me.DIAGNOSIS_CODES LIKE '%E761%' OR me.DIAGNOSIS_CODES LIKE '%E763%')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'  -- Exclude bad future dates
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.DIAGNOSIS_CODE IN ('E761','E763')
      AND pe.TRANSACTION_STATUS = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'  -- Exclude bad future dates
),

-- Get all treatment claims
all_tx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.NDC11 IN ('54092070001','540920700')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'  -- Exclude bad future dates
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.NDC11 IN ('54092070001','540920700')
      AND pe.TRANSACTION_RESULT = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'  -- Exclude bad future dates
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        me.RENDERING_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                '38206','38230','38232','38240','38241','38242','38243','38250')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'  -- Exclude bad future dates
),

-- Combine Dx + Tx
all_claims AS (
    SELECT patient_id, territory, claim_date, npi, 'DIAGNOSIS' AS claim_type
    FROM all_dx_claims
    UNION ALL
    SELECT patient_id, territory, claim_date, npi, 'TREATMENT' AS claim_type
    FROM all_tx_claims
),

-- Latest CLAIM (Dx or Tx) per patient - HCP-attributed (NPI not null)
latest_claim_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE npi IS NOT NULL
),
latest_claim_hcp AS (
    SELECT patient_id, territory, npi AS latest_claim_hcp_npi, claim_date AS latest_claim_date
    FROM latest_claim_hcp_ranked WHERE rn = 1
),

-- Latest CLAIM date (patient-level fallback, includes NULL NPI)
latest_claim_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_claim_date_any
    FROM all_claims
    GROUP BY patient_id, territory
),

-- Latest TREATMENT per patient - HCP-attributed (NPI not null)
latest_treatment_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
      AND npi IS NOT NULL
),
latest_treatment_hcp AS (
    SELECT patient_id, territory, npi AS latest_treatment_hcp_npi, claim_date AS latest_treatment_date
    FROM latest_treatment_hcp_ranked WHERE rn = 1
),

-- Latest TREATMENT date (patient-level fallback, includes NULL NPI)
latest_treatment_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_treatment_date_any
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
    GROUP BY patient_id, territory
),

-- Get all patients
all_patients AS (
    SELECT DISTINCT patient_id, territory FROM all_claims
)

-- Final output
SELECT 
    p.territory,
    p.patient_id,
    
    -- Latest claim (any type)
    COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
    lch.latest_claim_hcp_npi AS latest_claim_npi,
    
    -- Latest infusion/treatment
    COALESCE(lth.latest_treatment_date, ltd.latest_treatment_date_any) AS latest_infusion_date,
    lth.latest_treatment_hcp_npi AS latest_infusion_npi,
    
    -- Flag bad dates
    CASE 
        WHEN COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) >= '2025-11-30' THEN '⚠️ BAD FUTURE DATE'
        WHEN COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) > CURRENT_DATE THEN '⚠️ FUTURE'
        ELSE '✓ OK'
    END AS date_quality
    
FROM all_patients p
LEFT JOIN latest_claim_hcp lch ON p.patient_id = lch.patient_id
LEFT JOIN latest_claim_date_patient lcd ON p.patient_id = lcd.patient_id
LEFT JOIN latest_treatment_hcp lth ON p.patient_id = lth.patient_id
LEFT JOIN latest_treatment_date_patient ltd ON p.patient_id = ltd.patient_id

ORDER BY p.territory, latest_claim_date DESC, p.patient_id ASC;

In [0]:
-- =============================================================================
-- ALL CLAIMS for ALL Patients - Complete Claims Data
-- =============================================================================

WITH all_territory_patients AS (
    SELECT DISTINCT 
        patient_id,
        primary_hcp_territory_id_2yr AS territory
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
),

-- Get all diagnosis claims
all_dx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi,
        'MEDICAL_DX' AS claim_source,
        me.DIAGNOSIS_CODES
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE (me.DIAGNOSIS_CODES LIKE '%E761%' OR me.DIAGNOSIS_CODES LIKE '%E763%')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi,
        'PHARMACY_DX' AS claim_source,
        pe.DIAGNOSIS_CODE AS diagnosis_codes
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.DIAGNOSIS_CODE IN ('E761','E763')
      AND pe.TRANSACTION_STATUS = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
),

-- Get all treatment claims
all_tx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi,
        'MEDICAL_NDC' AS claim_source,
        me.NDC11 AS treatment_code
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.NDC11 IN ('54092070001','540920700')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi,
        'PHARMACY_NDC' AS claim_source,
        pe.NDC11 AS treatment_code
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.NDC11 IN ('54092070001','540920700')
      AND pe.TRANSACTION_RESULT = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        me.RENDERING_NPI AS npi,
        'MEDICAL_PROC' AS claim_source,
        me.PROCEDURE_CODE AS treatment_code
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                '38206','38230','38232','38240','38241','38242','38243','38250')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
)

-- Get all claims for ALL patients
SELECT 
    territory,
    patient_id,
    claim_date,
    npi,
    claim_source,
    'DIAGNOSIS' AS claim_category,
    diagnosis_codes,
    CAST(NULL AS STRING) AS treatment_code
FROM all_dx_claims

UNION ALL

SELECT 
    territory,
    patient_id,
    claim_date,
    npi,
    claim_source,
    'TREATMENT' AS claim_category,
    CAST(NULL AS STRING) AS diagnosis_codes,
    treatment_code
FROM all_tx_claims

ORDER BY 
    territory,
    patient_id,
    claim_date DESC;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.patient360_master;

In [0]:
-- =============================================================================
-- TERRITORY-LEVEL SWEEP: Latest Claim Dates for All Patients
-- Shows how filters affect latest dates across all territories
-- =============================================================================

WITH all_territory_patients AS (
    SELECT DISTINCT 
        patient_id,
        primary_hcp_territory_id_2yr AS territory
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- Get all diagnosis claims WITH SPECIALTY FILTER
all_dx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE (me.DIAGNOSIS_CODES LIKE '%E761%' OR me.DIAGNOSIS_CODES LIKE '%E763%')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
      AND (COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IN (SELECT npi FROM cohort_3_learnings) 
           OR COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IS NULL)
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.DIAGNOSIS_CODE IN ('E761','E763')
      AND pe.TRANSACTION_STATUS = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
      AND (pe.PRESCRIBER_NPI IN (SELECT npi FROM cohort_3_learnings) 
           OR pe.PRESCRIBER_NPI IS NULL)
),

-- Get all treatment claims WITH SPECIALTY FILTER
all_tx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.NDC11 IN ('54092070001','540920700')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
      AND (COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IN (SELECT npi FROM cohort_3_learnings) 
           OR COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IS NULL)
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.NDC11 IN ('54092070001','540920700')
      AND pe.TRANSACTION_RESULT = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
      AND (pe.PRESCRIBER_NPI IN (SELECT npi FROM cohort_3_learnings) 
           OR pe.PRESCRIBER_NPI IS NULL)
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        me.RENDERING_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                '38206','38230','38232','38240','38241','38242','38243','38250')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
      AND (me.RENDERING_NPI IN (SELECT npi FROM cohort_3_learnings) 
           OR me.RENDERING_NPI IS NULL)
),

-- Combine Dx + Tx
all_claims AS (
    SELECT patient_id, territory, claim_date, npi, 'DIAGNOSIS' AS claim_type
    FROM all_dx_claims
    UNION ALL
    SELECT patient_id, territory, claim_date, npi, 'TREATMENT' AS claim_type
    FROM all_tx_claims
),

-- Latest CLAIM (Dx or Tx) per patient - HCP-attributed (NPI not null)
latest_claim_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE npi IS NOT NULL
),
latest_claim_hcp AS (
    SELECT patient_id, territory, npi AS latest_claim_hcp_npi, claim_date AS latest_claim_date
    FROM latest_claim_hcp_ranked WHERE rn = 1
),

-- Latest CLAIM date (patient-level fallback, includes NULL NPI)
latest_claim_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_claim_date_any
    FROM all_claims
    GROUP BY patient_id, territory
),

-- Latest TREATMENT per patient - HCP-attributed (NPI not null)
latest_treatment_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
      AND npi IS NOT NULL
),
latest_treatment_hcp AS (
    SELECT patient_id, territory, npi AS latest_treatment_hcp_npi, claim_date AS latest_treatment_date
    FROM latest_treatment_hcp_ranked WHERE rn = 1
),

-- Latest TREATMENT date (patient-level fallback, includes NULL NPI)
latest_treatment_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_treatment_date_any
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
    GROUP BY patient_id, territory
),

-- Get all patients
all_patients AS (
    SELECT DISTINCT patient_id, territory FROM all_claims
),

-- Patient-level summary
patient_summary AS (
    SELECT 
        p.territory,
        p.patient_id,
        COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
        lch.latest_claim_hcp_npi AS latest_claim_npi,
        COALESCE(lth.latest_treatment_date, ltd.latest_treatment_date_any) AS latest_infusion_date,
        lth.latest_treatment_hcp_npi AS latest_infusion_npi
    FROM all_patients p
    LEFT JOIN latest_claim_hcp lch ON p.patient_id = lch.patient_id
    LEFT JOIN latest_claim_date_patient lcd ON p.patient_id = lcd.patient_id
    LEFT JOIN latest_treatment_hcp lth ON p.patient_id = lth.patient_id
    LEFT JOIN latest_treatment_date_patient ltd ON p.patient_id = ltd.patient_id
)

-- Territory-level rollup
SELECT 
    territory,
    COUNT(DISTINCT patient_id) AS total_patients,
    COUNT(DISTINCT CASE WHEN latest_claim_date > '2025-07-31' THEN patient_id END) AS patients_post_july,
    CAST(MIN(latest_claim_date) AS STRING) AS earliest_claim_date,
    CAST(MAX(latest_claim_date) AS STRING) AS latest_claim_date,
    CAST(MIN(latest_infusion_date) AS STRING) AS earliest_infusion_date,
    CAST(MAX(latest_infusion_date) AS STRING) AS latest_infusion_date
FROM patient_summary
GROUP BY territory
ORDER BY territory;

In [0]:
-- =============================================================================
-- TERRITORY-LEVEL SWEEP: Latest Claim Dates for All Patients
-- Shows how filters affect latest dates across all territories
-- =============================================================================

WITH all_territory_patients AS (
    SELECT DISTINCT 
        patient_id,
        primary_hcp_territory_id_2yr AS territory
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
),

cohort_3_learnings AS (
  SELECT DISTINCT npi
  FROM com_edp_prd.com_raw.kom_providers
  WHERE provider_type = 'INDIVIDUAL'
    AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
        'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology',
        'Radiology','Urology'
      )
      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine','Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine','Nutrition, Pediatric','Oncology, Pediatrics','Pediatric Cardiology',
        'Pediatric Critical Care Medicine','Pediatric Dermatology','Pediatric Emergency Medicine',
        'Pediatric Endocrinology','Pediatric Gastroenterology','Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases','Pediatric Nephrology','Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery','Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology',
        'Pediatric Rehabilitation Medicine','Pediatric Rheumatology','Pediatric Surgery','Pediatrics',
        'Clinical Biochemical Genetics','Clinical Genetics (M.D.)','Clinical Molecular Genetics',
        'Ph.D. Medical Genetics','Neurodevelopmental Disabilities','Neurology',
        'Neurology with Special Qualifications in Child Neurology','Neuroradiology'
      )
    )
),

-- Get all diagnosis claims WITH SPECIALTY FILTER
all_dx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE (me.DIAGNOSIS_CODES LIKE '%E761%' OR me.DIAGNOSIS_CODES LIKE '%E763%')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
      AND (COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IN (SELECT npi FROM cohort_3_learnings) 
           OR COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IS NULL)
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.DIAGNOSIS_CODE IN ('E761','E763')
      AND pe.TRANSACTION_STATUS = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
      AND (pe.PRESCRIBER_NPI IN (SELECT npi FROM cohort_3_learnings) 
           OR pe.PRESCRIBER_NPI IS NULL)
),

-- Get all treatment claims WITH SPECIALTY FILTER
all_tx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.NDC11 IN ('54092070001','540920700')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
      AND (COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IN (SELECT npi FROM cohort_3_learnings) 
           OR COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) IS NULL)
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.NDC11 IN ('54092070001','540920700')
      AND pe.TRANSACTION_RESULT = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
      AND (pe.PRESCRIBER_NPI IN (SELECT npi FROM cohort_3_learnings) 
           OR pe.PRESCRIBER_NPI IS NULL)
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        me.RENDERING_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                '38206','38230','38232','38240','38241','38242','38243','38250')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
      AND (me.RENDERING_NPI IN (SELECT npi FROM cohort_3_learnings) 
           OR me.RENDERING_NPI IS NULL)
),

-- Combine Dx + Tx
all_claims AS (
    SELECT patient_id, territory, claim_date, npi, 'DIAGNOSIS' AS claim_type
    FROM all_dx_claims
    UNION ALL
    SELECT patient_id, territory, claim_date, npi, 'TREATMENT' AS claim_type
    FROM all_tx_claims
),

-- Latest CLAIM (Dx or Tx) per patient - HCP-attributed (NPI not null)
latest_claim_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE npi IS NOT NULL
),
latest_claim_hcp AS (
    SELECT patient_id, territory, npi AS latest_claim_hcp_npi, claim_date AS latest_claim_date
    FROM latest_claim_hcp_ranked WHERE rn = 1
),

-- Latest CLAIM date (patient-level fallback, includes NULL NPI)
latest_claim_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_claim_date_any
    FROM all_claims
    GROUP BY patient_id, territory
),

-- Latest TREATMENT per patient - HCP-attributed (NPI not null)
latest_treatment_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
      AND npi IS NOT NULL
),
latest_treatment_hcp AS (
    SELECT patient_id, territory, npi AS latest_treatment_hcp_npi, claim_date AS latest_treatment_date
    FROM latest_treatment_hcp_ranked WHERE rn = 1
),

-- Latest TREATMENT date (patient-level fallback, includes NULL NPI)
latest_treatment_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_treatment_date_any
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
    GROUP BY patient_id, territory
),

-- Get all patients
all_patients AS (
    SELECT DISTINCT patient_id, territory FROM all_claims
),

-- Patient-level summary
patient_summary AS (
    SELECT 
        p.territory,
        p.patient_id,
        COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
        lch.latest_claim_hcp_npi AS latest_claim_npi,
        COALESCE(lth.latest_treatment_date, ltd.latest_treatment_date_any) AS latest_infusion_date,
        lth.latest_treatment_hcp_npi AS latest_infusion_npi
    FROM all_patients p
    LEFT JOIN latest_claim_hcp lch ON p.patient_id = lch.patient_id
    LEFT JOIN latest_claim_date_patient lcd ON p.patient_id = lcd.patient_id
    LEFT JOIN latest_treatment_hcp lth ON p.patient_id = lth.patient_id
    LEFT JOIN latest_treatment_date_patient ltd ON p.patient_id = ltd.patient_id
)

SELECT * FROM patient_summary;
-- Territory-level rollup
-- SELECT 
--     territory,
--     COUNT(DISTINCT patient_id) AS total_patients,
--     COUNT(DISTINCT CASE WHEN latest_claim_date > '2025-07-31' THEN patient_id END) AS patients_post_july,
--     CAST(MIN(latest_claim_date) AS STRING) AS earliest_claim_date,
--     CAST(MAX(latest_claim_date) AS STRING) AS latest_claim_date,
--     CAST(MIN(latest_infusion_date) AS STRING) AS earliest_infusion_date,
--     CAST(MAX(latest_infusion_date) AS STRING) AS latest_infusion_date
-- FROM patient_summary
-- GROUP BY territory
-- ORDER BY territory;

In [0]:
-- =============================================================================
-- TERRITORY-LEVEL SWEEP: Latest Claim Dates for All Patients
-- Shows how filters affect latest dates across all territories
-- =============================================================================

WITH all_territory_patients AS (
    SELECT DISTINCT 
        patient_id,
        primary_hcp_territory_id_2yr AS territory
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
),

-- Get all diagnosis claims WITHOUT SPECIALTY FILTER
all_dx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE (me.DIAGNOSIS_CODES LIKE '%E761%' OR me.DIAGNOSIS_CODES LIKE '%E763%')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.DIAGNOSIS_CODE IN ('E761','E763')
      AND pe.TRANSACTION_STATUS = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
),

-- Get all treatment claims WITHOUT SPECIALTY FILTER
all_tx_claims AS (
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        COALESCE(me.RENDERING_NPI, me.REFERRING_NPI) AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.NDC11 IN ('54092070001','540920700')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        pe.FILL_DATE AS claim_date,
        pe.PRESCRIBER_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_pharmacy_events pe
        ON t.patient_id = pe.PATIENT_ID
    WHERE pe.NDC11 IN ('54092070001','540920700')
      AND pe.TRANSACTION_RESULT = 'PAID'
      AND pe.FILL_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND pe.FILL_DATE < '2025-11-30'
    
    UNION ALL
    
    SELECT 
        t.patient_id,
        t.territory,
        me.SERVICE_DATE AS claim_date,
        me.RENDERING_NPI AS npi
    FROM all_territory_patients t
    INNER JOIN com_edp_prd.com_raw.kom_medical_events me
        ON t.patient_id = me.PATIENT_ID
    WHERE me.PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                '38206','38230','38232','38240','38241','38242','38243','38250')
      AND me.SERVICE_DATE BETWEEN '2020-08-01' AND '2026-12-31'
      AND me.SERVICE_DATE < '2025-11-30'
),

-- Combine Dx + Tx
all_claims AS (
    SELECT patient_id, territory, claim_date, npi, 'DIAGNOSIS' AS claim_type
    FROM all_dx_claims
    UNION ALL
    SELECT patient_id, territory, claim_date, npi, 'TREATMENT' AS claim_type
    FROM all_tx_claims
),

-- Latest CLAIM (Dx or Tx) per patient - HCP-attributed (NPI not null)
latest_claim_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE npi IS NOT NULL
),
latest_claim_hcp AS (
    SELECT patient_id, territory, npi AS latest_claim_hcp_npi, claim_date AS latest_claim_date
    FROM latest_claim_hcp_ranked WHERE rn = 1
),

-- Latest CLAIM date (patient-level fallback, includes NULL NPI)
latest_claim_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_claim_date_any
    FROM all_claims
    GROUP BY patient_id, territory
),

-- Latest TREATMENT per patient - HCP-attributed (NPI not null)
latest_treatment_hcp_ranked AS (
    SELECT
        patient_id,
        territory,
        npi,
        claim_date,
        ROW_NUMBER() OVER (
            PARTITION BY patient_id
            ORDER BY claim_date DESC, npi ASC
        ) AS rn
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
      AND npi IS NOT NULL
),
latest_treatment_hcp AS (
    SELECT patient_id, territory, npi AS latest_treatment_hcp_npi, claim_date AS latest_treatment_date
    FROM latest_treatment_hcp_ranked WHERE rn = 1
),

-- Latest TREATMENT date (patient-level fallback, includes NULL NPI)
latest_treatment_date_patient AS (
    SELECT patient_id, territory, MAX(claim_date) AS latest_treatment_date_any
    FROM all_claims
    WHERE claim_type = 'TREATMENT'
    GROUP BY patient_id, territory
),

-- Get all patients
all_patients AS (
    SELECT DISTINCT patient_id, territory FROM all_claims
),

-- Patient-level summary
patient_summary AS (
    SELECT 
        p.territory,
        p.patient_id,
        COALESCE(lch.latest_claim_date, lcd.latest_claim_date_any) AS latest_claim_date,
        lch.latest_claim_hcp_npi AS latest_claim_npi,
        COALESCE(lth.latest_treatment_date, ltd.latest_treatment_date_any) AS latest_infusion_date,
        lth.latest_treatment_hcp_npi AS latest_infusion_npi
    FROM all_patients p
    LEFT JOIN latest_claim_hcp lch ON p.patient_id = lch.patient_id
    LEFT JOIN latest_claim_date_patient lcd ON p.patient_id = lcd.patient_id
    LEFT JOIN latest_treatment_hcp lth ON p.patient_id = lth.patient_id
    LEFT JOIN latest_treatment_date_patient ltd ON p.patient_id = ltd.patient_id
)
SELECT * FROM patient_summary;

-- Territory-level rollup
-- SELECT 
--     territory,
--     COUNT(DISTINCT patient_id) AS total_patients,
--     COUNT(DISTINCT CASE WHEN latest_claim_date > '2025-07-31' THEN patient_id END) AS patients_post_july,
--     CAST(MIN(latest_claim_date) AS STRING) AS earliest_claim_date,
--     CAST(MAX(latest_claim_date) AS STRING) AS latest_claim_date,
--     CAST(MIN(latest_infusion_date) AS STRING) AS earliest_infusion_date,
--     CAST(MAX(latest_infusion_date) AS STRING) AS latest_infusion_date
-- FROM patient_summary
-- GROUP BY territory
-- ORDER BY territory;

In [0]:
-- Simple: All claims for K5HNZJ43 with pass/fail status

WITH all_claims AS (
    SELECT PATIENT_ID, SERVICE_DATE AS claim_date, COALESCE(RENDERING_NPI, REFERRING_NPI) AS npi, 'MEDICAL_DX' AS claim_source
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID = 'K5HNZJ43' AND (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
    
    UNION ALL
    
    SELECT PATIENT_ID, FILL_DATE AS claim_date, PRESCRIBER_NPI AS npi, 'PHARMACY_DX' AS claim_source
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID = 'K5HNZJ43' AND DIAGNOSIS_CODE IN ('E761','E763') AND TRANSACTION_STATUS = 'PAID'
    
    UNION ALL
    
    SELECT PATIENT_ID, SERVICE_DATE AS claim_date, COALESCE(RENDERING_NPI, REFERRING_NPI) AS npi, 'MEDICAL_TX' AS claim_source
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PATIENT_ID = 'K5HNZJ43' 
      AND (NDC11 IN ('54092070001','540920700')
           OR PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379',
                                 '38206','38230','38232','38240','38241','38242','38243','38250'))
    
    UNION ALL
    
    SELECT PATIENT_ID, FILL_DATE AS claim_date, PRESCRIBER_NPI AS npi, 'PHARMACY_TX' AS claim_source
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE PATIENT_ID = 'K5HNZJ43' AND NDC11 IN ('54092070001','540920700') AND TRANSACTION_RESULT = 'PAID'
),

allowed_npis AS (
    SELECT DISTINCT npi
    FROM com_edp_prd.com_raw.kom_providers
    WHERE provider_type = 'INDIVIDUAL'
      AND (PRIMARY_SPECIALTY NOT IN ('Anesthesiologist Assistant','Anesthesiology','Dentist','Dietitian, Registered',
           'Emergency Medical Technician, Basic','Emergency Medicine','General Acute Care Hospital',
           'Nurse Anesthetist, Certified Registered','Obstetrics & Gynecology','Pathology','Radiology','Urology')
           OR SECONDARY_SPECIALTY IN ('Child & Adolescent Psychiatry','Psychiatry','Adolescent Medicine',
           'Developmental - Behavioral Pediatrics','Neonatal-Perinatal Medicine','Nutrition, Pediatric',
           'Oncology, Pediatrics','Pediatric Cardiology','Pediatric Critical Care Medicine','Pediatric Dermatology',
           'Pediatric Emergency Medicine','Pediatric Endocrinology','Pediatric Gastroenterology',
           'Pediatric Hematology-Oncology','Pediatric Infectious Diseases','Pediatric Nephrology',
           'Pediatric Ophthalmology and Strabismus Specialist','Pediatric Orthopaedic Surgery',
           'Pediatric Otolaryngology','Pediatric Pulmonology','Pediatric Radiology','Pediatric Rehabilitation Medicine',
           'Pediatric Rheumatology','Pediatric Surgery','Pediatrics','Clinical Biochemical Genetics',
           'Clinical Genetics (M.D.)','Clinical Molecular Genetics','Ph.D. Medical Genetics',
           'Neurodevelopmental Disabilities','Neurology','Neurology with Special Qualifications in Child Neurology',
           'Neuroradiology'))
)

SELECT 
    claim_date,
    npi,
    claim_source,
    CASE 
        WHEN claim_date >= '2020-08-01' AND claim_date < '2025-11-30' 
         AND (npi IS NULL OR npi IN (SELECT npi FROM allowed_npis))
        THEN '✓ INCLUDED'
        ELSE '✗ EXCLUDED'
    END AS status

FROM all_claims
ORDER BY claim_date DESC;

In [0]:
SELECT * FROM patient360_base